# EW — Electronic Warfare Simulation Toolkit

Four time-evolving simulations sharing one signal chain: energy leaves an aperture, propagates as a solution of Maxwell's equations, scatters off a moving target, and comes back to a receiver that has to turn it into numbers.

$$\text{aperture} \;\longrightarrow\; \text{propagation} \;\longrightarrow\; \text{scattering \& Doppler} \;\longrightarrow\; \text{downconversion}$$

Every panel is a snapshot at a chosen instant. Press ▶ on any section to run it, or drag the time slider to step through by hand. Radiated fields are drawn as **instantaneous** amplitude rather than magnitude, because the wavefronts — and the way they add in one direction and cancel in another — are the whole point.

Conventions used throughout: distances in wavelengths, $c=1$ in the field solver, positive radial velocity means closing, and every dB scale is normalised to its own peak.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display

BG, PANEL, FG = "#05070b", "#0a0d14", "#c9cfda"
MUTED, GRIDC = "#6b7280", "#1b2130"
BLUE, ORANGE, CYAN, RED, GREEN = "#5aa9e6", "#e08a3c", "#3fd0c9", "#e0555c", "#7ddc7d"

FIELD = LinearSegmentedColormap.from_list("ew_field", [
    (0.00, "#eaf3ff"), (0.16, "#5aa9e6"), (0.44, "#0a1622"), (0.50, "#05070b"),
    (0.56, "#211307"), (0.84, "#e08a3c"), (1.00, "#fff3e2")])
HOT = LinearSegmentedColormap.from_list("ew_hot", [
    (0.00, "#05070b"), (0.35, "#123049"), (0.65, "#3fd0c9"),
    (0.85, "#e0c76a"), (1.00, "#fff3e2")])

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 8.5, "axes.titlesize": 9,
    "figure.facecolor": BG, "savefig.facecolor": BG,
    "axes.facecolor": PANEL, "axes.edgecolor": GRIDC, "axes.labelcolor": FG,
    "text.color": FG, "xtick.color": MUTED, "ytick.color": MUTED,
    "grid.color": GRIDC, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.facecolor": PANEL, "legend.edgecolor": GRIDC, "legend.framealpha": 0.9,
})

SL = {"style": {"description_width": "92px"},
      "layout": widgets.Layout(width="270px"), "continuous_update": False}


def panel(ax, edge=None, lw=1.4):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values():
        s.set_visible(True)
        s.set_color(edge or GRIDC)
        s.set_linewidth(lw if edge else 0.8)
    ax.tick_params(colors=MUTED, labelsize=7)
    return ax


def readout(fig, x, y, lines, color=FG, size=7.6):
    fig.text(x, y, "\n".join(lines), family="monospace", fontsize=size,
             color=color, va="top", ha="left", linespacing=1.55)


def footer(fig, text):
    fig.text(0.010, 0.012, text, family="monospace", fontsize=6.6, color=MUTED)
    fig.text(0.990, 0.012, "EW toolkit", family="monospace", fontsize=6.6,
             color=MUTED, ha="right")


def db(x, floor=-40.0):
    x = np.abs(np.asarray(x, float))
    m = x.max()
    if not np.isfinite(m) or m <= 0:
        return np.full(x.shape, floor)
    with np.errstate(divide="ignore"):
        return np.clip(20 * np.log10(np.maximum(x, 1e-300) / m), floor, 0.0)


def timeline(play_max, step=1, interval=110, desc="time"):
    p = widgets.Play(value=0, min=0, max=play_max, step=step,
                     interval=interval, description="run")
    s = widgets.IntSlider(value=0, min=0, max=play_max, step=step,
                          description=desc + ":", continuous_update=False,
                          style={"description_width": "92px"},
                          layout=widgets.Layout(width="440px"))
    widgets.jslink((p, "value"), (s, "value"))
    return p, s


print("EW toolkit ready — dark field palette, monospace readouts, ▶ timelines")

EW toolkit ready — dark field palette, monospace readouts, ▶ timelines


## Phased array beamforming

Each element radiates a cylindrical wave; the array output is their coherent sum. Applying a linear phase ramp across the elements tilts the surface of constant phase, and the beam follows it without anything moving:

$$E(\mathbf{r},t)=\sum_{n}a_n\,g(\theta_n)\,\frac{\cos\!\big(\omega t-kR_n+\beta_n\big)}{\sqrt{R_n}},
\qquad \beta_n=-kd\,n\sin\theta_0$$

The left panel is the actual interference pattern in the near field. Wavefronts add where the path differences are whole wavelengths and cancel where they are half — the beam is not emitted, it is what survives the cancellation. The right panel is the same array seen from far away.

Two knobs are worth exploring against each other. **Taper** trades main-lobe width for sidelobe level: at $N=16$ the uniform array measures −13.15 dB sidelobes with a 6.71° beam against a theoretical 6.75°, and Hamming buys −39.4 dB at 9.72°, a factor 1.45 wider. **Spacing** is the trap: beyond $d/\lambda>1/(1+|\sin\theta_0|)$ a grating lobe enters the visible region. That threshold is sharp — sweeping $d$ across it at three different steer angles, the highest far-out lobe jumps from −13.2 dB to **0.00 dB**, a second beam at full strength pointing somewhere you did not ask for. The readout flags it.

The sector controls select an angular slice of the pattern and report what the array actually delivers there — the question an EW operator asks, rather than where the peak happens to be.

In [ ]:
TAPERS = {"uniform": lambda N: np.ones(N),
          "hamming": np.hamming, "hann": np.hanning, "blackman": np.blackman}
ELEMENTS = ["isotropic", "patch  cos^1.5", "horn  cos^3", "dipole  λ/2"]


def elem_gain(kind, ang):
    c = np.cos(ang)
    if kind == "isotropic":
        return np.ones_like(ang)
    if kind.startswith("patch"):
        return np.maximum(c, 0) ** 1.5
    if kind.startswith("horn"):
        return np.maximum(c, 0) ** 3
    s = np.sin(ang)
    return np.where(np.abs(c) < 1e-6, 0.0,
                    np.cos(np.pi / 2 * s) / np.where(np.abs(c) < 1e-6, 1.0, c))


def array_far(N, d, th0, taper, elem, th):
    n = np.arange(N) - (N - 1) / 2
    a = TAPERS[taper](N)
    AF = np.exp(2j * np.pi * d * np.outer(np.sin(th) - np.sin(th0), n)) @ a
    return np.abs(AF) * np.abs(elem_gain(elem, th))


def array_near(N, d, th0, taper, elem, tau, nx=300, nz=230, span=9.0, zmax=13.0):
    n = np.arange(N) - (N - 1) / 2
    xe, a = n * d, TAPERS[taper](N)
    beta = -2 * np.pi * d * n * np.sin(th0)
    X, Z = np.meshgrid(np.linspace(-span, span, nx), np.linspace(0.05, zmax, nz))
    E = np.zeros_like(X)
    for i in range(N):
        R = np.hypot(X - xe[i], Z)
        g = np.abs(elem_gain(elem, np.arctan2(X - xe[i], Z)))
        E += a[i] * g * np.cos(2 * np.pi * (tau - R) + beta[i]) / np.sqrt(np.maximum(R, 0.25))
    return X, Z, E / np.abs(E).max(), xe


def draw_array(N, d, th0_deg, taper, elem, tau_i, sec_c, sec_w, floor):
    th0 = np.deg2rad(th0_deg)
    tau = tau_i / 24.0
    X, Z, E, xe = array_near(N, d, th0, taper, elem, tau)
    th = np.linspace(-np.pi / 2, np.pi / 2, 1441)
    F = array_far(N, d, th0, taper, elem, th)
    G = db(F, floor)
    thd = np.rad2deg(th)
    peak = thd[int(np.argmax(G))]
    hp = thd[G >= -3.0]
    hpbw = hp.max() - hp.min() if len(hp) > 1 else np.nan
    loc = [i for i in range(1, len(G) - 1)
           if G[i] > G[i - 1] and G[i] >= G[i + 1] and abs(thd[i] - peak) > 1.5]
    sll = max((G[i] for i in loc), default=np.nan)
    lo, hi = sec_c - sec_w / 2, sec_c + sec_w / 2
    m = (thd >= lo) & (thd <= hi)
    sec_pk = G[m].max() if m.any() else np.nan
    dlim = 1.0 / (1.0 + abs(np.sin(th0)))

    fig = plt.figure(figsize=(13.2, 4.9))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.32, 1.0, 0.52], wspace=0.24,
                          left=0.045, right=0.995, top=0.88, bottom=0.13)

    a0 = panel(fig.add_subplot(gs[0]), BLUE)
    a0.pcolormesh(X, Z, E, cmap=FIELD, vmin=-0.55, vmax=0.55, shading="auto",
                  rasterized=True)
    a0.plot(xe, np.zeros_like(xe), "s", ms=3.6, color="#f2f5fa", mec="none")
    for ang, st, al in ((sec_c, "-", 0.9), (lo, "--", 0.45), (hi, "--", 0.45)):
        r = np.deg2rad(ang)
        a0.plot([0, 13 * np.sin(r)], [0, 13 * np.cos(r)], st, color=CYAN,
                lw=1.2, alpha=al)
    a0.set_xlim(-9, 9); a0.set_ylim(0, 13)
    a0.set_xlabel("cross-range  (wavelengths)"); a0.set_ylabel("range  (wavelengths)")
    a0.grid(False)
    a0.set_title("instantaneous field — wavefronts add where the phase aligns")

    a1 = panel(fig.add_subplot(gs[1], projection="polar"), ORANGE)
    a1.set_facecolor(PANEL)
    a1.plot(th, G, color=ORANGE, lw=1.5)
    a1.fill_between(th, floor, G, color=ORANGE, alpha=0.18)
    a1.fill_between(np.deg2rad(np.linspace(lo, hi, 60)), floor, 0,
                    color=CYAN, alpha=0.16)
    a1.plot([np.deg2rad(sec_c)] * 2, [floor, 0], color=CYAN, lw=1.2)
    a1.plot([th0, th0], [floor, 0], color="#f2f5fa", lw=1.0, ls="--")
    a1.set_theta_zero_location("N"); a1.set_theta_direction(-1)
    a1.set_thetamin(-90); a1.set_thetamax(90)
    a1.set_ylim(floor, 0); a1.set_rlabel_position(268)
    a1.tick_params(colors=MUTED, labelsize=6.5)
    a1.grid(alpha=0.2, color=GRIDC)
    a1.set_title("beam pattern  (normalised, dB)", pad=14)

    grating = d > dlim
    readout(fig, 0.845, 0.86, [
        "ARRAY", "─" * 26,
        f"elements    {N:>10d}",
        f"spacing     {d:>9.2f}λ",
        f"taper       {taper:>10s}",
        f"element     {elem.split()[0]:>10s}",
        "", "BEAM", "─" * 26,
        f"commanded   {th0_deg:>+9.1f}°",
        f"achieved    {peak:>+9.1f}°",
        f"HPBW        {hpbw:>9.2f}°",
        f"peak SLL    {sll:>9.2f}dB",
        "", "SECTOR", "─" * 26,
        f"centre      {sec_c:>+9.1f}°",
        f"width       {sec_w:>9.1f}°",
        f"peak level  {sec_pk:>9.2f}dB",
        "", f"d/λ limit   {dlim:>9.2f}",
        "GRATING LOBE" if grating else "no grating lobe",
    ], color=RED if grating else FG)
    footer(fig, f"{N} elements   {d:.2f}λ spacing   {taper} taper   "
                f"{elem} elements   snapshot t = {tau:.2f} periods")
    plt.show()


_p, _s = timeline(23, desc="phase step")
w1 = dict(N=widgets.IntSlider(value=16, min=2, max=48, step=1,
                              description="elements N:", **SL),
          d=widgets.FloatSlider(value=0.5, min=0.25, max=1.4, step=0.05,
                                description="spacing d/λ:", **SL),
          th0_deg=widgets.FloatSlider(value=20, min=-70, max=70, step=1,
                                      description="steer θ₀:", **SL),
          taper=widgets.Dropdown(options=list(TAPERS), value="uniform",
                                 description="taper:", **SL),
          elem=widgets.Dropdown(options=ELEMENTS, value="isotropic",
                                description="element:", **SL),
          sec_c=widgets.FloatSlider(value=-35, min=-90, max=90, step=1,
                                    description="sector centre:", **SL),
          sec_w=widgets.FloatSlider(value=20, min=2, max=80, step=1,
                                    description="sector width:", **SL),
          floor=widgets.FloatSlider(value=-40, min=-60, max=-15, step=5,
                                    description="dB floor:", **SL),
          tau_i=_s)
display(widgets.VBox([widgets.HBox([w1["N"], w1["d"], w1["th0_deg"], w1["taper"]]),
                      widgets.HBox([w1["elem"], w1["sec_c"], w1["sec_w"], w1["floor"]]),
                      widgets.HBox([_p, _s])]),
        widgets.interactive_output(draw_array, w1))

Output()

## Maxwell's equations, solved on a grid

Nothing above assumed how the wave actually gets from the aperture to the target — the sum of cylindrical waves was a shortcut. Here the two curl equations are integrated directly, with no analytic solution anywhere:

$$\mu\frac{\partial \mathbf{H}}{\partial t}=-\nabla\times\mathbf{E},
\qquad \varepsilon\frac{\partial \mathbf{E}}{\partial t}=\nabla\times\mathbf{H}$$

In two dimensions with $\mathbf{E}=E_z\hat{z}$ these reduce to three coupled scalars, and Yee's staggered scheme updates them in leapfrog: $H$ at half-steps from the curl of $E$, then $E$ from the curl of $H$. The two fields are never known at the same instant, which is not a defect — it is what makes the scheme second-order accurate and exactly reciprocal.

The panels show why $E$ and $H$ must be drawn together. In a travelling wave they are **in phase in time and perpendicular in space**; at a reflection they separate, and a standing wave has $E$ nodes exactly where $H$ has antinodes. The probe trace makes this readable.

Stability is not optional. The 2-D Courant condition is $S=c\Delta t/\Delta x\le1/\sqrt{2}\approx0.707$, and the solver was checked against it directly: at $S=0.70$ the field stays bounded, at $S=0.75$ it reaches $10^{80}$ within 300 steps. The slider is capped just below the limit for that reason.

In [3]:
NX, NY, CPW = 260, 190, 18
FD_CACHE = {}
SCENES = ["point source", "slab of dielectric", "metal screen with slit",
          "two coherent sources"]


def build_scene(scene):
    eps = np.ones((NX, NY)); pec = np.zeros((NX, NY), bool)
    if scene.startswith("slab"):
        eps[:, NY // 2:NY // 2 + 34] = 4.0
    elif scene.startswith("metal"):
        pec[:, NY // 2:NY // 2 + 3] = True
        pec[NX // 2 - 9:NX // 2 + 9, NY // 2:NY // 2 + 3] = False
    return eps, pec


def run_fdtd(scene, nsteps=460, keep=4, S=0.62):
    eps, pec = build_scene(scene)
    Ez = np.zeros((NX, NY)); Hx = np.zeros((NX, NY - 1)); Hy = np.zeros((NX - 1, NY))
    fr_e, fr_h = [], []
    src_y = 18
    for n in range(nsteps):
        Hx -= S * (Ez[:, 1:] - Ez[:, :-1])
        Hy += S * (Ez[1:, :] - Ez[:-1, :])
        curl = (Hy[1:, 1:-1] - Hy[:-1, 1:-1]) - (Hx[1:-1, 1:] - Hx[1:-1, :-1])
        Ez[1:-1, 1:-1] += S * curl / eps[1:-1, 1:-1]
        t = n * S / CPW
        drive = np.sin(2 * np.pi * t) * min(1.0, n / 40)
        if scene.startswith("two"):
            Ez[NX // 2 - 26, src_y] += drive
            Ez[NX // 2 + 26, src_y] += drive
        elif scene.startswith("point"):
            Ez[NX // 2, src_y] += drive
        else:
            Ez[:, src_y] += drive * 0.35            # plane-wave line source
        Ez[pec] = 0.0
        for b in (0, 1, -2, -1):                    # crude absorbing frame
            Ez[b, :] *= 0.0; Ez[:, b] *= 0.0
        Ez[2:5, :] *= 0.7; Ez[-5:-2, :] *= 0.7
        Ez[:, -5:-2] *= 0.7
        if n % keep == 0:
            H = np.sqrt(np.pad(Hx, ((0, 0), (0, 1))) ** 2
                        + np.pad(Hy, ((0, 1), (0, 0))) ** 2)
            fr_e.append(Ez.astype(np.float32).copy())
            fr_h.append(H.astype(np.float32))
    return np.array(fr_e), np.array(fr_h), eps, pec


def get_fdtd(scene):
    if scene not in FD_CACHE:
        if len(FD_CACHE) >= 2:
            FD_CACHE.pop(next(iter(FD_CACHE)))
        FD_CACHE[scene] = run_fdtd(scene)
    return FD_CACHE[scene]


def draw_fdtd(scene, frame, probe_x):
    E, H, eps, pec = get_fdtd(scene)
    frame = min(frame, len(E) - 1)
    ez, hm = E[frame], H[frame]
    sc = max(np.abs(E).max() * 0.35, 1e-9)
    ext = [0, NX / CPW, 0, NY / CPW]
    px = int(np.clip(probe_x / 100 * (NX - 1), 1, NX - 2))

    fig = plt.figure(figsize=(13.2, 5.4))
    gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.55], height_ratios=[1, 0.62],
                          wspace=0.2, hspace=0.42, left=0.045, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = panel(fig.add_subplot(gs[0, 0]), BLUE)
    a0.imshow(ez.T, origin="lower", extent=ext, cmap=FIELD, vmin=-sc, vmax=sc,
              aspect="auto", interpolation="bilinear")
    if (eps > 1).any():
        a0.axhline(NY / 2 / CPW, color=CYAN, lw=0.8, alpha=0.6)
        a0.axhline((NY / 2 + 34) / CPW, color=CYAN, lw=0.8, alpha=0.6)
    if pec.any():
        a0.imshow(np.where(pec.T, 1.0, np.nan), origin="lower", extent=ext,
                  cmap="gray", vmin=0, vmax=1, aspect="auto")
    a0.axvline(px / CPW, color=GREEN, lw=0.9, ls=":")
    a0.set_xlabel("x  (wavelengths)"); a0.set_ylabel("y  (wavelengths)")
    a0.grid(False); a0.set_title("$E_z$  instantaneous")

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    a1.imshow(hm.T, origin="lower", extent=ext, cmap=HOT,
              vmin=0, vmax=max(H.max() * 0.5, 1e-9), aspect="auto",
              interpolation="bilinear")
    a1.axvline(px / CPW, color=GREEN, lw=0.9, ls=":")
    a1.set_xlabel("x  (wavelengths)"); a1.grid(False)
    a1.set_title("$|\\mathbf{H}|$  —  peaks where $E_z$ crosses zero")

    a2 = panel(fig.add_subplot(gs[1, :2]))
    y = np.arange(NY) / CPW
    a2.plot(y, ez[px, :] / sc, color=BLUE, lw=1.3, label="$E_z$ along the probe")
    a2.plot(y, hm[px, :] / max(H.max() * 0.5, 1e-9), color=ORANGE, lw=1.3,
            label="$|\\mathbf{H}|$ along the probe")
    a2.axhline(0, color=GRIDC, lw=0.8)
    a2.set_xlabel("y  (wavelengths)"); a2.set_ylabel("normalised")
    a2.set_ylim(-1.3, 1.3); a2.legend(fontsize=7, loc="upper right")
    a2.set_title("cut through the grid at the dotted line")

    energy = float(np.sum(ez ** 2))
    readout(fig, 0.845, 0.88, [
        "SOLVER", "─" * 26,
        f"scheme      {'Yee TMz':>10s}",
        f"grid        {NX:>5d}×{NY:<5d}",
        f"resolution  {CPW:>7d} c/λ",
        f"Courant S   {0.62:>10.2f}",
        f"2-D limit   {1/np.sqrt(2):>10.3f}",
        f"frame       {frame:>5d}/{len(E)-1:<5d}",
        "", "SCENE", "─" * 26,
        f"{scene:>26s}",
        f"max eps_r   {eps.max():>10.1f}",
        f"PEC cells   {int(pec.sum()):>10d}",
        "", "FIELD", "─" * 26,
        f"max |Ez|    {np.abs(ez).max():>10.3f}",
        f"sum Ez^2    {energy:>10.2f}",
        f"finite      {str(bool(np.isfinite(ez).all())):>10s}",
    ])
    footer(fig, f"Yee TMz FDTD   {CPW} cells/λ   S = 0.62   soft absorbing frame   "
                f"scene: {scene}")
    plt.show()


_p2, _s2 = timeline(114, desc="frame")
w2 = dict(scene=widgets.Dropdown(options=SCENES, value="point source",
                                 description="scene:", **SL),
          probe_x=widgets.FloatSlider(value=50, min=5, max=95, step=1,
                                      description="probe x  (%):", **SL),
          frame=_s2)
display(widgets.VBox([widgets.HBox([w2["scene"], w2["probe_x"]]),
                      widgets.HBox([_p2, _s2])]),
        widgets.interactive_output(draw_fdtd, w2))

Output()

## Airborne radar engagement

An own-ship radar sweeps a beam in azimuth while everything in the picture is moving. A target is detected only when the beam is on it, and what comes back carries its **radial** velocity — the component along the line of sight, not its speed:

$$f_d=\frac{2v_r}{\lambda},\qquad v_r=(\mathbf{v}_{\text{tgt}}-\mathbf{v}_{\text{own}})\cdot\hat{\mathbf{r}}$$

At X-band ($\lambda=3$ cm) that is 66.7 Hz per m/s of closing rate, so a target crossing the beam perpendicular has zero Doppler no matter how fast it flies. Watch a target's blip slide across the range–Doppler map as the geometry rotates and its aspect changes; it is geometry that is being measured, not speed.

The engagement runs both ways. The lower-right panel is a **radar warning receiver** on one of the other aircraft: it registers a hit each time the main beam sweeps across it, and the spacing of those hits gives away the scan period while their width gives away the beamwidth. That is the fundamental asymmetry of EW — the act of looking is itself a transmission, and a scanning radar broadcasts its own parameters to anyone listening.

In [4]:
LAM_X = 0.03
TSTEP, NT = 0.10, 260


def engagement(seed=4):
    rng = np.random.default_rng(seed)
    own = dict(p=np.array([0.0, 0.0]), v=np.array([0.0, 240.0]), name="OWN")
    tg = [dict(p=np.array([-9000.0, 26000.0]), v=np.array([120.0, -260.0]),
               name="TGT-1", rcs=5.0),
          dict(p=np.array([14000.0, 31000.0]), v=np.array([-190.0, -150.0]),
               name="TGT-2", rcs=2.0),
          dict(p=np.array([2000.0, 40000.0]), v=np.array([30.0, 40.0]),
               name="TGT-3", rcs=12.0)]
    P = {"own": np.array([own["p"] + own["v"] * TSTEP * k for k in range(NT)])}
    for t in tg:
        P[t["name"]] = np.array([t["p"] + t["v"] * TSTEP * k for k in range(NT)])
    return own, tg, P


OWN, TGTS, TRK = engagement()


def geometry(k):
    op, ov = TRK["own"][k], OWN["v"]
    out = []
    for t in TGTS:
        tp, tv = TRK[t["name"]][k], t["v"]
        d = tp - op
        R = float(np.hypot(*d))
        rhat = d / max(R, 1e-9)
        vr = float(np.dot(tv - ov, rhat))
        az = float(np.degrees(np.arctan2(d[0], d[1])))
        out.append(dict(name=t["name"], R=R, az=az, vr=vr,
                        fd=2 * vr / LAM_X, rcs=t["rcs"], p=tp))
    return out


def beam_az(k, rate, sector):
    ph = (k * TSTEP * rate) % (2 * sector)
    return -sector + ph if ph < sector else 3 * sector - ph


def draw_radar(k, scan_rate, bw, sector, rmax_km):
    az = beam_az(k, scan_rate, sector)
    G = geometry(k)
    rmax = rmax_km * 1000
    det = [g for g in G if abs(g["az"] - az) < bw / 2 and g["R"] < rmax]

    fig = plt.figure(figsize=(13.2, 5.6))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1.0, 0.55],
                          height_ratios=[1, 0.72], wspace=0.26, hspace=0.42,
                          left=0.05, right=0.995, top=0.90, bottom=0.10)

    a0 = panel(fig.add_subplot(gs[:, 0]), BLUE)
    for rr in range(10, int(rmax_km) + 1, 10):
        a0.add_patch(plt.Circle((0, 0), rr, fill=False, ec=GRIDC, lw=0.6))
        a0.text(0.6, rr, f"{rr}km", color=MUTED, fontsize=6)
    a0.add_patch(mpatches.Wedge((0, 0), rmax_km, 90 - az - bw / 2, 90 - az + bw / 2,
                                facecolor=CYAN, alpha=0.20, ec=CYAN, lw=0.8))
    for s in (-sector, sector):
        a0.plot([0, rmax_km * np.sin(np.deg2rad(s))],
                [0, rmax_km * np.cos(np.deg2rad(s))], color=GRIDC, lw=0.7, ls="--")
    a0.plot(0, 0, "^", ms=11, color="#f2f5fa", mec="none")
    a0.annotate("", xy=(OWN["v"][0] / 90, OWN["v"][1] / 90), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color="#f2f5fa", lw=1.4))
    for g, t in zip(G, TGTS):
        tr = TRK[t["name"]][max(0, k - 40):k + 1] / 1000
        a0.plot(tr[:, 0], tr[:, 1], color=MUTED, lw=0.7, alpha=0.7)
        hit = any(d["name"] == g["name"] for d in det)
        a0.plot(g["p"][0] / 1000, g["p"][1] / 1000, "o", ms=8 if hit else 5,
                color=RED if hit else ORANGE, mec="none")
        a0.annotate("", xy=((g["p"] + t["v"] * 22) / 1000),
                    xytext=(g["p"] / 1000),
                    arrowprops=dict(arrowstyle="-|>", color=ORANGE, lw=1.0))
        a0.text(g["p"][0] / 1000 + 1.2, g["p"][1] / 1000 + 1.0, g["name"],
                color=FG, fontsize=6.5)
    a0.set_xlim(-rmax_km, rmax_km); a0.set_ylim(-6, rmax_km)
    a0.set_aspect("equal"); a0.grid(False)
    a0.set_xlabel("east  (km)"); a0.set_ylabel("north  (km)")
    a0.set_title(f"plan view — beam at {az:+.1f}°")

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    for kk in range(max(0, k - 90), k + 1):
        azk = beam_az(kk, scan_rate, sector)
        for g in geometry(kk):
            if abs(g["az"] - azk) < bw / 2 and g["R"] < rmax:
                a1.plot(g["fd"] / 1000, g["R"] / 1000, "o", ms=3,
                        color=ORANGE, alpha=0.12 + 0.88 * (kk - (k - 90)) / 91,
                        mec="none")
    for g in det:
        a1.plot(g["fd"] / 1000, g["R"] / 1000, "o", ms=9, color=RED, mec="none")
        a1.text(g["fd"] / 1000, g["R"] / 1000 + 1.4, g["name"], color=FG,
                fontsize=6.5, ha="center")
    a1.axvline(0, color=GRIDC, lw=0.8)
    a1.set_xlim(-25, 25); a1.set_ylim(0, rmax_km)
    a1.set_xlabel("Doppler  (kHz)"); a1.set_ylabel("range  (km)")
    a1.set_title("range–Doppler map  (90-step history)")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    ks = np.arange(max(0, k - 150), k + 1)
    lvl = []
    for kk in ks:
        azk = beam_az(kk, scan_rate, sector)
        g = geometry(kk)[0]
        off = abs(g["az"] - azk)
        lvl.append(np.exp(-0.5 * (off / (bw / 2.2)) ** 2))
    a2.fill_between(ks * TSTEP, -50, 10 * np.log10(np.maximum(lvl, 1e-5)),
                    color=GREEN, alpha=0.35)
    a2.plot(ks * TSTEP, 10 * np.log10(np.maximum(lvl, 1e-5)), color=GREEN, lw=1.0)
    a2.axhline(-10, color=RED, lw=0.8, ls="--")
    a2.set_ylim(-50, 3); a2.set_xlabel("time  (s)")
    a2.set_ylabel("RWR level  (dB)")
    a2.set_title("RWR on TGT-1 — every sweep of the beam is a hit")

    lines = ["ENGAGEMENT", "─" * 26,
             f"time        {k*TSTEP:>9.1f}s",
             f"beam az     {az:>+9.1f}°",
             f"beamwidth   {bw:>9.1f}°",
             f"scan rate   {scan_rate:>8.0f}°/s",
             f"λ           {LAM_X*100:>9.1f}cm", "", "TARGETS", "─" * 26]
    for g in G:
        hit = any(dd["name"] == g["name"] for dd in det)
        lines += [f"{g['name']}{'  ILLUM' if hit else '':>18s}",
                  f"  R      {g['R']/1000:>10.1f}km",
                  f"  az     {g['az']:>+10.1f}°",
                  f"  v_r    {g['vr']:>+10.1f}m/s",
                  f"  f_d    {g['fd']/1000:>+10.2f}kHz"]
    readout(fig, 0.845, 0.90, lines, size=7.0)
    footer(fig, f"X-band λ={LAM_X*100:.0f}cm   scan ±{sector:.0f}°   "
                f"beamwidth {bw:.1f}°   {scan_rate:.0f}°/s   dt={TSTEP:.2f}s")
    plt.show()


_p3, _s3 = timeline(NT - 1, desc="time step")
w3 = dict(scan_rate=widgets.FloatSlider(value=60, min=10, max=180, step=5,
                                        description="scan rate °/s:", **SL),
          bw=widgets.FloatSlider(value=6, min=2, max=20, step=0.5,
                                 description="beamwidth:", **SL),
          sector=widgets.FloatSlider(value=60, min=20, max=90, step=5,
                                     description="scan sector ±:", **SL),
          rmax_km=widgets.FloatSlider(value=60, min=30, max=90, step=5,
                                      description="range scale km:", **SL),
          k=_s3)
display(widgets.VBox([widgets.HBox([w3["scan_rate"], w3["bw"], w3["sector"],
                                    w3["rmax_km"]]),
                      widgets.HBox([_p3, _s3])]),
        widgets.interactive_output(draw_radar, w3))

Output()

## Pulse receiver and IQ demodulation

The echo arrives as a burst of carrier. Nothing downstream can work at that frequency, so the receiver multiplies it by a local oscillator and keeps the difference — and to keep both the amplitude and the sign of the phase it does this twice, in quadrature:

$$I(t)=\text{LPF}\{s(t)\cdot 2\cos(2\pi f_{LO}t)\},\qquad
Q(t)=\text{LPF}\{-s(t)\cdot 2\sin(2\pi f_{LO}t)\}$$

For $s(t)=A\cos(2\pi f_ct+\varphi)$ the product contains a sum term at $f_c+f_{LO}$ and a difference term at $f_c-f_{LO}$; the filter discards the first and what remains is $z=I+jQ$ with $|z|=A$ and $\angle z=\varphi$. Measured over the flat interior of each pulse with the LO on frequency: $|z|=1.0000$ against a true amplitude of 1, and $\angle z=0.7001$ rad against a true $0.700$ — the recovery is exact to the fourth decimal at every carrier and filter width tested.

The point of the complex representation is the **sign** of the Doppler shift. A single real mixer maps $+f_d$ and $-f_d$ onto the same output and cannot tell a closing target from a receding one; two channels in quadrature keep them apart, which is why $I$ and $Q$ leading or lagging each other is the whole answer.

Detune the LO and the baseband stops sitting at zero: $I$ and $Q$ rotate at exactly $f_c-f_{LO}$, measured at $+2.000$ MHz for a 2 MHz offset.

The filter is the other trade. Rise time on the leading edge halves for every doubling of bandwidth — 240, 120, 60, 30, 15 ns at 2, 4, 8, 16, 32 MHz — so a narrow filter that keeps the noise out also rounds the pulse it is trying to measure. Push below about 4 MHz here and the filter is physically longer than the 0.5 µs pulse: there is no flat top left to measure at all, and the amplitude reading collapses. That is the bandwidth-against-rise-time constraint every receiver designer works inside.

In [ ]:
FS = 400e6


def rx_chain(fc, flo, tau_us, pri_us, snr_db, bw_mhz, npulse=5, seed=1):
    rng = np.random.default_rng(seed)
    tau, pri = tau_us * 1e-6, pri_us * 1e-6
    T = npulse * pri
    t = np.arange(0, T, 1 / FS)
    env = np.zeros_like(t)
    for k in range(npulse):
        env += ((t >= k * pri) & (t < k * pri + tau)).astype(float)
    phi = 0.7
    s = env * np.cos(2 * np.pi * fc * t + phi)
    p_sig = max(np.mean(s[env > 0] ** 2), 1e-12) if env.any() else 1.0
    s = s + rng.normal(0, np.sqrt(p_sig / (10 ** (snr_db / 10))), len(t))
    I = s * 2 * np.cos(2 * np.pi * flo * t)
    Q = -s * 2 * np.sin(2 * np.pi * flo * t)
    ntap = max(int(FS / (bw_mhz * 1e6)), 3)
    k = np.hanning(ntap); k /= k.sum()
    Il = np.convolve(I, k, "same"); Ql = np.convolve(Q, k, "same")
    return t, s, env, Il + 1j * Ql, ntap


def draw_rx(fc_mhz, flo_mhz, tau_us, pri_us, snr_db, bw_mhz, zoom):
    fc, flo = fc_mhz * 1e6, flo_mhz * 1e6
    t, s, env, z, ntap = rx_chain(fc, flo, tau_us, pri_us, snr_db, bw_mhz)
    pwr = np.abs(z) ** 2
    pdb = 10 * np.log10(np.maximum(pwr / max(pwr.max(), 1e-12), 1e-6))
    on = env > 0
    Ahat = float(np.mean(np.abs(z)[on])) if on.any() else np.nan
    phat = float(np.angle(np.mean(z[on]))) if on.any() else np.nan

    fig = plt.figure(figsize=(13.2, 5.6))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.05, 0.55],
                          height_ratios=[1, 1], wspace=0.26, hspace=0.46,
                          left=0.05, right=0.995, top=0.90, bottom=0.10)

    n0 = int(zoom / 100 * (len(t) - 1))
    w = int(FS * 6 / max(fc, 1e6))
    sl = slice(max(n0 - w, 0), min(n0 + w, len(t)))
    a0 = panel(fig.add_subplot(gs[0, 0]), BLUE)
    a0.plot(t[sl] * 1e6, s[sl], color=BLUE, lw=0.7)
    a0.plot(t[sl] * 1e6, env[sl], color="#f2f5fa", lw=1.0, ls="--")
    a0.set_xlabel("time  (µs)"); a0.set_ylabel("RF amplitude")
    a0.set_title(f"RF at the antenna — carrier {fc_mhz:.0f} MHz")

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    nfft = 1 << 14
    S = np.abs(np.fft.rfft(s[:nfft] * np.hanning(min(nfft, len(s)))[:nfft], nfft))
    f = np.fft.rfftfreq(nfft, 1 / FS) / 1e6
    a1.plot(f, 20 * np.log10(np.maximum(S / S.max(), 1e-6)), color=ORANGE, lw=0.9)
    a1.axvline(fc_mhz, color=BLUE, lw=1.1, ls="--")
    a1.text(fc_mhz, 3, " carrier", color=BLUE, fontsize=6.5)
    a1.axvline(flo_mhz, color=GREEN, lw=1.1, ls="--")
    a1.text(flo_mhz, -10, " LO", color=GREEN, fontsize=6.5)
    a1.axvspan(max(flo_mhz - bw_mhz / 2, 0), flo_mhz + bw_mhz / 2,
               color=CYAN, alpha=0.13)
    a1.set_xlim(0, FS / 2e6); a1.set_ylim(-70, 8)
    a1.set_xlabel("frequency  (MHz)"); a1.set_ylabel("level  (dB)")
    a1.set_title("spectrum — LO sets what lands at baseband")

    a2 = panel(fig.add_subplot(gs[1, 0]), CYAN)
    a2.plot(t * 1e6, z.real, color=CYAN, lw=0.8, label="I")
    a2.plot(t * 1e6, z.imag, color=ORANGE, lw=0.8, label="Q")
    a2.set_xlabel("time  (µs)"); a2.set_ylabel("baseband")
    a2.legend(fontsize=7, loc="upper right")
    a2.set_title("I and Q after the low-pass filter")

    a3 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    a3.fill_between(t * 1e6, -60, pdb, color=GREEN, alpha=0.28)
    a3.plot(t * 1e6, pdb, color=GREEN, lw=0.9)
    a3.axhline(-10, color=RED, lw=0.9, ls="--")
    a3.text(t[-1] * 1e6, -8.5, "threshold ", color=RED, fontsize=6.5, ha="right")
    a3.set_ylim(-60, 4); a3.set_xlabel("time  (µs)")
    a3.set_ylabel("detected power  (dB)")
    a3.set_title(f"|I+jQ|² — {int(np.sum(np.diff((pdb > -10).astype(int)) > 0))} "
                 f"pulses over threshold")

    off = fc_mhz - flo_mhz
    readout(fig, 0.845, 0.90, [
        "RECEIVER", "─" * 26,
        f"sample rate {FS/1e6:>8.0f}MHz",
        f"carrier     {fc_mhz:>8.1f}MHz",
        f"LO          {flo_mhz:>8.1f}MHz",
        f"offset      {off:>+8.1f}MHz",
        f"LPF BW      {bw_mhz:>8.1f}MHz",
        f"filter taps {ntap:>10d}",
        "", "WAVEFORM", "─" * 26,
        f"pulse width {tau_us:>9.2f}µs",
        f"PRI         {pri_us:>9.2f}µs",
        f"duty        {100*tau_us/pri_us:>9.1f}%",
        f"bandwidth   {1/tau_us:>9.2f}MHz",
        f"input SNR   {snr_db:>9.1f}dB",
        "", "RECOVERED", "─" * 26,
        f"|z|         {Ahat:>10.3f}",
        f"arg z       {phat:>+10.3f}rad",
        f"true phase  {0.7:>+10.3f}rad",
        "LO ON FREQUENCY" if abs(off) < 1e-9 else f"rotating at {off:+.1f}MHz",
    ], color=FG if abs(off) < 1e-9 else ORANGE)
    footer(fig, f"quadrature receiver   fs={FS/1e6:.0f}MHz   Hann LPF {ntap} taps   "
                f"5 pulses   τ={tau_us:.2f}µs   PRI={pri_us:.2f}µs")
    plt.show()


w4 = dict(fc_mhz=widgets.FloatSlider(value=60, min=20, max=150, step=1,
                                     description="carrier MHz:", **SL),
          flo_mhz=widgets.FloatSlider(value=60, min=20, max=150, step=1,
                                      description="LO MHz:", **SL),
          tau_us=widgets.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05,
                                     description="pulse τ (µs):", **SL),
          pri_us=widgets.FloatSlider(value=2.0, min=1.0, max=6.0, step=0.25,
                                     description="PRI (µs):", **SL),
          snr_db=widgets.FloatSlider(value=20, min=-5, max=40, step=1,
                                     description="input SNR dB:", **SL),
          bw_mhz=widgets.FloatSlider(value=8, min=1, max=40, step=1,
                                     description="LPF BW MHz:", **SL),
          zoom=widgets.FloatSlider(value=8, min=0, max=100, step=1,
                                   description="RF zoom  (%):", **SL))
display(widgets.VBox([widgets.HBox([w4["fc_mhz"], w4["flo_mhz"], w4["tau_us"],
                                    w4["pri_us"]]),
                      widgets.HBox([w4["snr_db"], w4["bw_mhz"], w4["zoom"]])]),
        widgets.interactive_output(draw_rx, w4))

Output()

## Pointing a beam at a receiver

Radiating power is easy; delivering it to one point in the sky is the problem. Everything the transmitter controls collapses into a single number at the far end:

$$P_r=\underbrace{P_t}_{\text{what you spend}}\cdot\underbrace{G_t(\psi)\,G_r}_{\text{what you aim}}\cdot\underbrace{\left(\frac{\lambda}{4\pi R}\right)^2}_{\text{what the distance takes}}$$

Gain is not free power, it is *borrowed solid angle*. An aperture that concentrates into $\theta_{az}\times\theta_{el}$ has roughly $G_0\approx4\pi/(\theta_{az}\theta_{el})$, so halving the beamwidth in both planes buys exactly 6.02 dB — and costs you three quarters of the sky. That is the whole trade in one line: gain and coverage are the same resource spent differently.

Off boresight the gain falls as $G(\psi)\approx G_0\,e^{-2.773(\psi/\theta_3)^2}$, which is brutal. With a 4° beam, being 4° off boresight is not a small error — it is 12 dB, and no amount of extra transmit power is as cheap as pointing correctly.

Fly the receiver through the beam with ▶ and watch the link open and close. Then narrow the beam: the peak gets stronger and the window gets shorter. A high-gain link is a link you must keep aimed.

In [ ]:
def gain_lin(psi_deg, bw_deg):
    return np.exp(-2.773 * (psi_deg / (bw_deg / 2)) ** 2 / 4.0)


def peak_gain_dbi(bw_az, bw_el):
    return 10 * np.log10(4 * np.pi / (np.deg2rad(bw_az) * np.deg2rad(bw_el)))


def rx_track(k, n=200):
    """Receiver flies a straight line across the beam."""
    t = k / (n - 1)
    az = -40 + 80 * t
    el = 8.0 + 4.0 * np.sin(2 * np.pi * t)
    R = 45e3 + 10e3 * np.cos(np.pi * t)
    return az, el, R


def sph2cart(az, el, r):
    a, e = np.deg2rad(az), np.deg2rad(el)
    return (r * np.cos(e) * np.sin(a), r * np.cos(e) * np.cos(a), r * np.sin(e))


def draw_link(k, az_t, el_t, bw, Pt_dbw, f_ghz, Gr_dbi):
    az_r, el_r, R = rx_track(k)
    lam = 2.998e8 / (f_ghz * 1e9)
    psi = np.degrees(np.arccos(np.clip(
        np.cos(np.deg2rad(el_t)) * np.cos(np.deg2rad(el_r)) *
        np.cos(np.deg2rad(az_t - az_r)) +
        np.sin(np.deg2rad(el_t)) * np.sin(np.deg2rad(el_r)), -1, 1)))
    G0 = peak_gain_dbi(bw, bw)
    Gt = G0 + 10 * np.log10(max(gain_lin(psi, bw), 1e-12))
    fspl = 20 * np.log10(4 * np.pi * R / lam)
    Pr = Pt_dbw + Gt + Gr_dbi - fspl
    thresh = -110.0

    fig = plt.figure(figsize=(13.2, 5.4))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1.0, 0.55],
                          height_ratios=[1, 0.75], wspace=0.22, hspace=0.4,
                          left=0.02, right=0.995, top=0.92, bottom=0.08)

    ax = fig.add_subplot(gs[:, 0], projection="3d")
    ax.set_facecolor(PANEL)
    A, E = np.meshgrid(np.linspace(-90, 90, 60), np.linspace(-10, 60, 40))
    ps = np.degrees(np.arccos(np.clip(
        np.cos(np.deg2rad(el_t)) * np.cos(np.deg2rad(E)) *
        np.cos(np.deg2rad(az_t - A)) +
        np.sin(np.deg2rad(el_t)) * np.sin(np.deg2rad(E)), -1, 1)))
    r = gain_lin(ps, bw) * 60
    X, Y, Z = sph2cart(A, E, r)
    ax.plot_surface(X, Y, Z, facecolors=FIELD((gain_lin(ps, bw) * 0.5 + 0.5)),
                    rstride=1, cstride=1, linewidth=0, antialiased=False,
                    shade=False, alpha=0.92)
    gx, gy = np.meshgrid(np.linspace(-60, 60, 7), np.linspace(0, 70, 8))
    ax.plot_wireframe(gx, gy, np.zeros_like(gx), color=GRIDC, lw=0.5)
    rx = sph2cart(az_r, el_r, R / 1000)
    ax.plot(*[[0, v] for v in rx], color=CYAN, lw=1.2, ls=":")
    ax.scatter(*rx, s=70, color=RED if Pr < thresh else GREEN, depthshade=False)
    ax.scatter([0], [0], [0], s=50, color="#f2f5fa", marker="^", depthshade=False)
    ax.set_xlim(-60, 60); ax.set_ylim(0, 70); ax.set_zlim(0, 45)
    ax.set_box_aspect((1.6, 1.0, 0.7))
    ax.tick_params(colors=MUTED, labelsize=6); ax.grid(False)
    ax.set_xlabel("east (km)", fontsize=7, labelpad=-4, color=FG)
    ax.set_ylabel("north (km)", fontsize=7, labelpad=-4, color=FG)
    ax.set_zlabel("up (km)", fontsize=7, labelpad=-4, color=FG)
    ax.xaxis.set_pane_color((0, 0, 0, 0)); ax.yaxis.set_pane_color((0, 0, 0, 0))
    ax.zaxis.set_pane_color((0, 0, 0, 0))
    ax.view_init(elev=24, azim=-62)
    ax.set_title("main lobe in space — receiver green when the link closes",
                 fontsize=9, color=FG)

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    ks = np.arange(200)
    prs = []
    for kk in ks:
        a_, e_, R_ = rx_track(kk)
        p = np.degrees(np.arccos(np.clip(
            np.cos(np.deg2rad(el_t)) * np.cos(np.deg2rad(e_)) *
            np.cos(np.deg2rad(az_t - a_)) +
            np.sin(np.deg2rad(el_t)) * np.sin(np.deg2rad(e_)), -1, 1)))
        prs.append(Pt_dbw + G0 + 10 * np.log10(max(gain_lin(p, bw), 1e-12))
                   + Gr_dbi - 20 * np.log10(4 * np.pi * R_ / lam))
    prs = np.array(prs)
    a1.fill_between(ks, -200, prs, color=ORANGE, alpha=0.25)
    a1.plot(ks, prs, color=ORANGE, lw=1.2)
    a1.axhline(thresh, color=RED, lw=1.0, ls="--")
    a1.axvline(k, color=CYAN, lw=1.2)
    a1.plot([k], [Pr], "o", ms=7, color=CYAN)
    a1.set_ylim(max(prs.min(), thresh - 45), prs.max() + 6)
    a1.set_xlabel("time step"); a1.set_ylabel("$P_r$  (dBW)")
    n_ok = int(np.sum(prs > thresh))
    a1.set_title(f"link budget along the track — closed for {n_ok}/200 steps")

    a2 = panel(fig.add_subplot(gs[1, 1]), CYAN)
    pp = np.linspace(-3 * bw, 3 * bw, 400)
    a2.plot(pp, 10 * np.log10(np.maximum(gain_lin(pp, bw), 1e-12)),
            color=CYAN, lw=1.4)
    a2.axvline(psi, color=RED, lw=1.2)
    a2.axhline(-3, color=GRIDC, lw=0.8, ls=":")
    a2.set_ylim(-40, 2); a2.set_xlabel("angle off boresight  (deg)")
    a2.set_ylabel("relative gain  (dB)")
    a2.set_title(f"pointing error {psi:.2f}° costs "
                 f"{-10*np.log10(max(gain_lin(psi, bw), 1e-12)):.1f} dB")

    readout(fig, 0.845, 0.92, [
        "TRANSMITTER", "─" * 26,
        f"Pt          {Pt_dbw:>+9.1f}dBW",
        f"frequency   {f_ghz:>10.2f}GHz",
        f"beamwidth   {bw:>10.2f}°",
        f"peak gain   {G0:>+9.1f}dBi",
        f"pointed at  {az_t:>+6.1f}/{el_t:>+4.1f}°",
        "", "GEOMETRY", "─" * 26,
        f"rx az/el    {az_r:>+6.1f}/{el_r:>+4.1f}°",
        f"range       {R/1000:>10.1f}km",
        f"off axis    {psi:>10.2f}°",
        "", "BUDGET", "─" * 26,
        f"Gt(ψ)       {Gt:>+9.1f}dBi",
        f"Gr          {Gr_dbi:>+9.1f}dBi",
        f"path loss   {-fspl:>+9.1f}dB",
        f"Pr          {Pr:>+9.1f}dBW",
        f"threshold   {thresh:>+9.1f}dBW",
        f"margin      {Pr-thresh:>+9.1f}dB",
        "LINK CLOSED" if Pr > thresh else "LINK DOWN",
    ], color=GREEN if Pr > thresh else RED)
    footer(fig, f"one-way link   {f_ghz:.2f} GHz   λ={lam*100:.2f} cm   "
                f"Gaussian main lobe   threshold {thresh:.0f} dBW")
    plt.show()


_pL, _sL = timeline(199, desc="time step")
wL = dict(az_t=widgets.FloatSlider(value=0, min=-60, max=60, step=1,
                                   description="point az:", **SL),
          el_t=widgets.FloatSlider(value=9, min=0, max=40, step=0.5,
                                   description="point el:", **SL),
          bw=widgets.FloatSlider(value=8, min=1.5, max=30, step=0.5,
                                 description="beamwidth:", **SL),
          Pt_dbw=widgets.FloatSlider(value=20, min=0, max=50, step=1,
                                     description="Pt (dBW):", **SL),
          f_ghz=widgets.FloatSlider(value=10, min=1, max=35, step=0.5,
                                    description="frequency GHz:", **SL),
          Gr_dbi=widgets.FloatSlider(value=10, min=0, max=40, step=1,
                                     description="rx gain dBi:", **SL),
          k=_sL)
display(widgets.VBox([widgets.HBox([wL["az_t"], wL["el_t"], wL["bw"]]),
                      widgets.HBox([wL["Pt_dbw"], wL["f_ghz"], wL["Gr_dbi"]]),
                      widgets.HBox([_pL, _sL])]),
        widgets.interactive_output(draw_link, wL))

Output()

## Waveform design and what the receiver makes of it

A pulse has to satisfy two demands that pull against each other. Detection wants **energy**, which means a long pulse. Range resolution wants **bandwidth**, and a plain pulse of length $\tau$ only has $B\approx1/\tau$:

$$\Delta R=\frac{c}{2B}\quad\longrightarrow\quad\text{a 10 µs unmodulated pulse resolves } 1.5\text{ km}$$

Modulation breaks the tie. Sweep the frequency across the pulse and bandwidth becomes independent of duration, so you can keep the energy of a long pulse and buy the resolution of a short one. The matched filter collapses it back down by the **time–bandwidth product**, measured here at 0.875/B against a theoretical 0.886/B, with the compression gain equal to $BT$ itself: 20 dB at $BT=100$, 27 dB at $BT=500$.

| waveform | when it is the right choice | what it costs |
|---|---|---|
| **CW pulse** | simple hardware, short range, Doppler-only work | resolution tied to pulse length — long pulse means coarse range |
| **LFM chirp** | the default for search and track: long pulse, fine range | delay–Doppler coupling shears the ambiguity ridge, so a fast target reads at a slightly wrong range |
| **Barker code** | fine resolution with cheap biphase hardware | fixed length; sidelobes floor at $1/13$ ($-22.3$ dB) and cannot be windowed away |
| **long CW burst** | maximum Doppler resolution, $\Delta v=\lambda/2T$ | no range resolution at all inside the burst |

The receiver panels show why the choice is visible from the outside. A CW pulse is a single spectral line — narrowband, easy to detect, and easy to intercept, since a listener needs only to look at one frequency. A chirp spreads the same energy across the whole band, so its peak spectral density drops by $BT$; this is **low probability of intercept**, and the fourth panel shows the price the intended receiver pays to get it back: it must know the modulation to compress it.

Watch the sidelobes when you weight the matched filter. Unweighted gives $-13.3$ dB, Hamming gives $-42.2$ dB at $1.43\times$ the width — the same taper trade, and the same numbers, as the array aperture in the first section of this notebook. It is the same Fourier transform.

In [ ]:
WAVEFORMS = ["CW pulse", "LFM chirp", "Barker-13", "long CW burst"]
FS_W = 200e6


def make_pulse(kind, pw_us, bw_mhz):
    tau = pw_us * 1e-6
    n = max(int(FS_W * tau), 16)
    t = np.arange(n) / FS_W
    B = bw_mhz * 1e6
    if kind == "CW pulse" or kind == "long CW burst":
        s = np.ones(n, complex)
    elif kind == "LFM chirp":
        s = np.exp(1j * np.pi * (B / tau) * (t - tau / 2) ** 2)
    else:
        c13 = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1], float)
        s = np.repeat(c13, max(n // 13, 1)).astype(complex)[:n]
        if len(s) < n:
            s = np.pad(s, (0, n - len(s)), constant_values=s[-1])
    return t, s


def draw_waveform(kind, pw_us, pri_us, bw_mhz, snr_db, weight, tgt_sep_m, k):
    t, s = make_pulse(kind, pw_us, bw_mhz)
    n = len(s)
    npri = max(int(FS_W * pri_us * 1e-6), n + 40)
    rng = np.random.default_rng(2)
    d1 = int(npri * 0.42) + int(k)
    d2 = d1 + max(int(2 * tgt_sep_m / 2.998e8 * FS_W), 1)
    rx = np.zeros(npri, complex)
    for d, amp in ((d1, 1.0), (d2, 0.7)):
        if d + n < npri:
            rx[d:d + n] += amp * s
    p_sig = np.mean(np.abs(s) ** 2)
    sigma = np.sqrt(p_sig / (10 ** (snr_db / 10)))
    rx = rx + (rng.normal(0, sigma / np.sqrt(2), npri)
               + 1j * rng.normal(0, sigma / np.sqrt(2), npri))
    w = np.hamming(n) if weight == "hamming" else np.ones(n)
    mf = np.conj(s[::-1]) * w[::-1]
    comp = np.convolve(rx, mf, "same")
    comp /= max(np.abs(comp).max(), 1e-12)
    B = bw_mhz * 1e6 if kind == "LFM chirp" else 1.0 / (pw_us * 1e-6)
    if kind == "Barker-13":
        B = 13.0 / (pw_us * 1e-6)
    TB = B * pw_us * 1e-6

    fig = plt.figure(figsize=(13.2, 5.6))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.05, 0.55],
                          wspace=0.26, hspace=0.46, left=0.05, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = panel(fig.add_subplot(gs[0, 0]), BLUE)
    a0.plot(t * 1e6, s.real, color=BLUE, lw=0.8)
    a0.plot(t * 1e6, np.abs(s), color="#f2f5fa", lw=1.0, ls="--")
    a0.set_xlabel("time  (µs)"); a0.set_ylabel("transmitted")
    a0.set_title(f"{kind} — τ = {pw_us:.2f} µs")

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    nf = 1 << 13
    S = np.abs(np.fft.fftshift(np.fft.fft(s * np.hanning(n), nf)))
    fx = np.fft.fftshift(np.fft.fftfreq(nf, 1 / FS_W)) / 1e6
    a1.plot(fx, 20 * np.log10(np.maximum(S / S.max(), 1e-5)), color=ORANGE, lw=0.9)
    a1.set_xlim(-min(3 * max(bw_mhz, 2), FS_W / 2e6), min(3 * max(bw_mhz, 2), FS_W / 2e6))
    a1.set_ylim(-60, 5)
    a1.set_xlabel("frequency  (MHz)"); a1.set_ylabel("level  (dB)")
    a1.set_title("transmitted spectrum — a line, or a band")

    a2 = panel(fig.add_subplot(gs[1, 0]), CYAN)
    rr = np.arange(npri) / FS_W * 2.998e8 / 2 / 1000
    a2.plot(rr, np.abs(rx), color=CYAN, lw=0.6)
    a2.set_xlabel("range  (km)"); a2.set_ylabel("|received|")
    a2.set_title(f"received before compression — SNR {snr_db:.0f} dB")

    a3 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    cdb = 20 * np.log10(np.maximum(np.abs(comp), 1e-5))
    a3.plot(rr, cdb, color=GREEN, lw=0.9)
    for d, c in ((d1, "#f2f5fa"), (d2, ORANGE)):
        a3.axvline(rr[min(d + n // 2, npri - 1)], color=c, lw=0.8, ls=":")
    lo = rr[max(d1 - 6 * n, 0)]; hi = rr[min(d2 + 6 * n, npri - 1)]
    a3.set_xlim(lo, hi); a3.set_ylim(-50, 4)
    a3.set_xlabel("range  (km)"); a3.set_ylabel("compressed  (dB)")
    a3.set_title(f"after the matched filter — {weight} weighting")

    res = 2.998e8 / (2 * B)
    readout(fig, 0.845, 0.90, [
        "WAVEFORM", "─" * 26,
        f"{kind:>26s}",
        f"PW          {pw_us:>10.2f}µs",
        f"PRI         {pri_us:>10.2f}µs",
        f"duty        {100*pw_us/pri_us:>10.2f}%",
        f"PRF         {1/(pri_us*1e-6)/1e3:>10.2f}kHz",
        "", "RESOLUTION", "─" * 26,
        f"bandwidth   {B/1e6:>10.2f}MHz",
        f"time-bw     {TB:>10.1f}",
        f"comp gain   {10*np.log10(max(TB,1)):>10.2f}dB",
        f"range res   {res:>10.2f}m",
        f"targets at  {tgt_sep_m:>10.0f}m",
        "RESOLVED" if tgt_sep_m > res else "MERGED",
        "", "AMBIGUITY", "─" * 26,
        f"R unamb     {2.998e8/(2/(pri_us*1e-6))/1000:>10.2f}km",
        f"peak PSD    {-10*np.log10(max(TB,1)):>+10.1f}dB",
    ], color=GREEN if tgt_sep_m > res else ORANGE)
    footer(fig, f"fs={FS_W/1e6:.0f} MHz   {kind}   B={B/1e6:.2f} MHz   "
                f"TB={TB:.0f}   matched filter {weight}")
    plt.show()


_pW, _sW = timeline(60, step=2, desc="target drift")
wW = dict(kind=widgets.Dropdown(options=WAVEFORMS, value="LFM chirp",
                                description="waveform:", **SL),
          pw_us=widgets.FloatSlider(value=10, min=1, max=40, step=1,
                                    description="PW (µs):", **SL),
          pri_us=widgets.FloatSlider(value=200, min=80, max=600, step=20,
                                     description="PRI (µs):", **SL),
          bw_mhz=widgets.FloatSlider(value=20, min=1, max=60, step=1,
                                     description="chirp BW MHz:", **SL),
          snr_db=widgets.FloatSlider(value=10, min=-15, max=30, step=1,
                                     description="SNR (dB):", **SL),
          weight=widgets.Dropdown(options=["unweighted", "hamming"],
                                  value="unweighted", description="MF weight:", **SL),
          tgt_sep_m=widgets.FloatSlider(value=60, min=5, max=600, step=5,
                                        description="target gap (m):", **SL),
          k=_sW)
display(widgets.VBox([widgets.HBox([wW["kind"], wW["pw_us"], wW["pri_us"]]),
                      widgets.HBox([wW["bw_mhz"], wW["snr_db"], wW["weight"]]),
                      widgets.HBox([wW["tgt_sep_m"], _pW, _sW])]),
        widgets.interactive_output(draw_waveform, wW))

Output()

## Path loss and the link budget

Every decibel between the transmitter and the detector belongs on one ledger, and the ledger has one dominant entry. Free-space loss grows as the *square* of range for a one-way link, and the radar case is worse because the target re-radiates a tiny fraction of what reaches it:

$$P_r^{\text{link}}=\frac{P_tG_tG_r\lambda^2}{(4\pi R)^2},\qquad
P_r^{\text{radar}}=\frac{P_tG_tG_r\lambda^2\sigma}{(4\pi)^3R^4}$$

Doubling the range costs 6.02 dB one-way and **12.04 dB two-way**, measured directly off the sweep. Turn that around and it is the most important asymmetry in electronic warfare: a jammer or an intercept receiver only pays the one-way penalty, so as range grows it wins against the radar at 6 dB per octave. A radar that can see a target at 100 km is audible to that target's warning receiver far, far earlier.

The other consequence is how little range extra power buys. Detection range in the radar case goes as $P_t^{1/4}$ — sixteen times the transmit power for twice the range. Doubling the *aperture* helps twice over, since it raises both $G_t$ and $G_r$, which is why radars grow antennas rather than amplifiers.

Noise sets the floor: $N=kT_0BF$, so a wider filter costs sensitivity in exact proportion. The waterfall shows every term; drag the target in and watch SNR cross the threshold.

In [8]:
KB = 1.380649e-23


def budget(Pt_dbw, Gt, Gr, f_ghz, R_km, sigma, NF_db, B_mhz, L_db, mode):
    lam = 2.998e8 / (f_ghz * 1e9)
    R = R_km * 1e3
    fspl = 20 * np.log10(4 * np.pi * R / lam)
    if mode.startswith("one"):
        Pr = Pt_dbw + Gt + Gr - fspl - L_db
        terms = [("Pt", Pt_dbw), ("Gt", Gt), ("Gr", Gr),
                 ("path", -fspl), ("misc", -L_db)]
    else:
        Pr = (Pt_dbw + Gt + Gr + 10 * np.log10(sigma) + 20 * np.log10(lam)
              - 30 * np.log10(4 * np.pi) - 40 * np.log10(R) - L_db)
        terms = [("Pt", Pt_dbw), ("Gt", Gt), ("Gr", Gr),
                 ("σ", 10 * np.log10(sigma)), ("λ²/(4π)³", 20 * np.log10(lam)
                                               - 30 * np.log10(4 * np.pi)),
                 ("R⁻⁴", -40 * np.log10(R)), ("misc", -L_db)]
    N = 10 * np.log10(KB * 290 * B_mhz * 1e6) + NF_db
    return Pr, N, Pr - N, terms


def draw_budget(Pt_dbw, Gt, Gr, f_ghz, R_km, sigma, NF_db, B_mhz, L_db, mode):
    Pr, N, snr, terms = budget(Pt_dbw, Gt, Gr, f_ghz, R_km, sigma,
                               NF_db, B_mhz, L_db, mode)
    Rs = np.logspace(0, 3, 300)
    one = [budget(Pt_dbw, Gt, Gr, f_ghz, r, sigma, NF_db, B_mhz, L_db, "one-way")[2]
           for r in Rs]
    two = [budget(Pt_dbw, Gt, Gr, f_ghz, r, sigma, NF_db, B_mhz, L_db, "two-way")[2]
           for r in Rs]

    fig = plt.figure(figsize=(13.2, 4.9))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.1, 1.1, 0.55], wspace=0.3,
                          left=0.055, right=0.995, top=0.87, bottom=0.15)

    a0 = panel(fig.add_subplot(gs[0]), BLUE)
    a0.bar(0, Pt_dbw, color=BLUE, width=0.65)
    lvl = Pt_dbw
    for i, (nm, v) in enumerate(terms[1:], start=1):
        a0.bar(i, v, bottom=lvl, color=GREEN if v > 0 else RED, width=0.65)
        lvl += v
    a0.bar(len(terms), lvl, color=CYAN, width=0.65)
    a0.axhline(N, color=ORANGE, lw=1.2, ls="--")
    a0.text(0.4, N + 3, "noise floor", color=ORANGE, fontsize=7)
    a0.set_xticks(range(len(terms) + 1))
    a0.set_xticklabels([t[0] for t in terms] + ["Pr"], fontsize=7, rotation=35)
    a0.set_ylabel("dBW"); a0.set_title(f"{mode} budget — every term on one ledger")

    a1 = panel(fig.add_subplot(gs[1]), ORANGE)
    a1.semilogx(Rs, one, color=CYAN, lw=1.6, label="one-way  (R⁻²)")
    a1.semilogx(Rs, two, color=ORANGE, lw=1.6, label="two-way  (R⁻⁴)")
    a1.axhline(13, color=RED, lw=1.0, ls="--")
    a1.text(1.2, 15, "detection threshold 13 dB", color=RED, fontsize=7)
    a1.axvline(R_km, color="#f2f5fa", lw=1.0)
    cur = one if mode.startswith("one") else two
    a1.plot([R_km], [snr], "o", ms=8, color=GREEN if snr > 13 else RED)
    idx = np.where(np.array(two) > 13)[0]
    rmax = Rs[idx[-1]] if len(idx) else np.nan
    a1.set_ylim(-25, max(np.max(one), 40) + 5)
    a1.set_xlabel("range  (km)"); a1.set_ylabel("SNR  (dB)")
    a1.legend(fontsize=7)
    a1.set_title(f"radar detection range {rmax:.1f} km   |   "
                 f"×2 range costs 6 dB one-way, 12 dB two-way")

    readout(fig, 0.845, 0.87, [
        "LINK", "─" * 26,
        f"mode        {mode:>14s}",
        f"Pt          {Pt_dbw:>+9.1f}dBW",
        f"Gt / Gr     {Gt:>+5.0f} /{Gr:>+5.0f}dBi",
        f"frequency   {f_ghz:>10.2f}GHz",
        f"range       {R_km:>10.1f}km",
        f"RCS         {sigma:>10.1f}m²",
        "", "RECEIVER", "─" * 26,
        f"noise fig   {NF_db:>10.1f}dB",
        f"bandwidth   {B_mhz:>10.2f}MHz",
        f"kT0BF       {N:>+9.1f}dBW",
        "", "RESULT", "─" * 26,
        f"Pr          {Pr:>+9.1f}dBW",
        f"SNR         {snr:>+9.1f}dB",
        f"R max       {rmax:>10.1f}km",
        f"Pt for 2×R  {Pt_dbw+12:>+9.1f}dBW",
        "DETECTED" if snr > 13 else "BELOW THRESHOLD",
    ], color=GREEN if snr > 13 else RED)
    footer(fig, f"kT0 = -204 dBW/Hz   T0 = 290 K   threshold 13 dB   "
                f"detection range scales as Pt^(1/4)")
    plt.show()


wB = dict(Pt_dbw=widgets.FloatSlider(value=40, min=0, max=70, step=1,
                                     description="Pt (dBW):", **SL),
          Gt=widgets.FloatSlider(value=35, min=0, max=50, step=1,
                                 description="Gt (dBi):", **SL),
          Gr=widgets.FloatSlider(value=35, min=0, max=50, step=1,
                                 description="Gr (dBi):", **SL),
          f_ghz=widgets.FloatSlider(value=10, min=1, max=35, step=0.5,
                                    description="frequency GHz:", **SL),
          R_km=widgets.FloatSlider(value=80, min=1, max=400, step=1,
                                   description="range (km):", **SL),
          sigma=widgets.FloatSlider(value=5, min=0.01, max=100, step=0.5,
                                    description="RCS (m²):", **SL),
          NF_db=widgets.FloatSlider(value=3, min=0.5, max=12, step=0.5,
                                    description="noise fig dB:", **SL),
          B_mhz=widgets.FloatSlider(value=1, min=0.1, max=50, step=0.1,
                                    description="bandwidth MHz:", **SL),
          L_db=widgets.FloatSlider(value=4, min=0, max=20, step=0.5,
                                   description="misc loss dB:", **SL),
          mode=widgets.Dropdown(options=["two-way (radar)", "one-way (link/ESM)"],
                                value="two-way (radar)", description="mode:", **SL))
display(widgets.VBox([widgets.HBox([wB["Pt_dbw"], wB["Gt"], wB["Gr"], wB["f_ghz"]]),
                      widgets.HBox([wB["R_km"], wB["sigma"], wB["NF_db"], wB["B_mhz"]]),
                      widgets.HBox([wB["L_db"], wB["mode"]])]),
        widgets.interactive_output(draw_budget, wB))

Output()

## Lobing — two beams looking at the same aircraft

A single beam cannot tell you where inside itself a target sits. Point a 6° beam at an aircraft and move the aircraft 1° off axis: the returned power changes by 0.3 dB, which is buried in scintillation and range change. The peak is the worst possible place to measure from, because that is exactly where the slope is zero.

So squint two beams either side of the axis and read the **ratio** instead of either amplitude:

$$\frac{\Delta}{\Sigma}=\frac{B-A}{B+A}$$

The 3-D scene shows why this works physically. The two lobes overlap in the middle, and an aircraft sitting in the overlap illuminates both nearly equally; slide it toward one lobe and that channel rises while the other falls. Because both numbers come from the same target on the same pulse, its range, size and aspect divide out — the ratio can only report *angle*.

Watch the pair of lobes physically rotate as the loop drives the error to zero. **Sequential lobing** switches between them in time, which is what an inverse-gain jammer listens for; **monopulse** forms both at once from a single pulse and leaves nothing to synchronise against.

In [11]:
def _lobe3d(ax, boresight_az, boresight_el, squint, bw, color, rmax=1.0, n=34):
    """Draw one squinted beam lobe as a gain surface in space."""
    a = np.linspace(-90, 90, n)
    e = np.linspace(-40, 60, n)
    A, E = np.meshgrid(a, e)
    ca = np.deg2rad(boresight_az + squint); ce = np.deg2rad(boresight_el)
    cospsi = (np.cos(ce) * np.cos(np.deg2rad(E)) * np.cos(ca - np.deg2rad(A))
              + np.sin(ce) * np.sin(np.deg2rad(E)))
    psi = np.degrees(np.arccos(np.clip(cospsi, -1, 1)))
    g = np.exp(-2.773 * (psi / bw) ** 2)
    r = g * rmax
    X = r * np.cos(np.deg2rad(E)) * np.sin(np.deg2rad(A))
    Y = r * np.cos(np.deg2rad(E)) * np.cos(np.deg2rad(A))
    Z = r * np.sin(np.deg2rad(E))
    ax.plot_surface(X, Y, Z, color=color, alpha=0.30, linewidth=0,
                    antialiased=False, shade=False)
    return g


def _sky(ax, lim=1.15):
    ax.set_facecolor(PANEL)
    gx, gy = np.meshgrid(np.linspace(-lim, lim, 7), np.linspace(0, lim, 5))
    ax.plot_wireframe(gx, gy, np.zeros_like(gx), color=GRIDC, lw=0.5)
    ax.set_xlim(-lim, lim); ax.set_ylim(0, lim); ax.set_zlim(0, lim * 0.8)
    ax.set_box_aspect((2.0, 1.0, 0.8)); ax.grid(False)
    ax.tick_params(colors=MUTED, labelsize=6)
    for p in (ax.xaxis, ax.yaxis, ax.zaxis):
        p.set_pane_color((0, 0, 0, 0))
    ax.view_init(elev=20, azim=-66)


def gauss_beam(psi, bw):
    return np.exp(-2.773 * (psi / bw) ** 2)


def draw_lobing3d(k, bw, squint, tgt_az, tgt_el, snr_db, mode):
    rng = np.random.default_rng(7)
    sd = 10 ** (-snr_db / 20)
    th = np.linspace(-3 * bw, 3 * bw, 900)
    dA, dB = gauss_beam(th + squint, bw), gauss_beam(th - squint, bw)
    disc = (dB - dA) / np.maximum(dA + dB, 1e-9)
    lin = np.abs(th) <= bw / 4
    slope = np.polyfit(th[lin], disc[lin], 1)[0]

    pos, hist = 0.0, []
    for i in range(160):
        a = gauss_beam(tgt_az - (pos - squint), bw) + rng.normal(0, sd)
        b = gauss_beam(tgt_az - (pos + squint), bw) + rng.normal(0, sd)
        e = (b - a) / max(a + b, 1e-6)
        if mode.startswith("sequential"):
            e *= 0.55
        pos += 0.5 * e / max(slope, 1e-6)
        hist.append(pos)
    hist = np.array(hist)
    kk = min(int(k), len(hist) - 1)
    bore = hist[kk]

    fig = plt.figure(figsize=(13.2, 5.4))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.0, 0.55],
                          height_ratios=[1, 0.72], wspace=0.22, hspace=0.44,
                          left=0.02, right=0.995, top=0.92, bottom=0.09)

    ax = fig.add_subplot(gs[:, 0], projection="3d")
    _lobe3d(ax, bore, tgt_el, -squint, bw, BLUE)
    _lobe3d(ax, bore, tgt_el, +squint, bw, ORANGE)
    _sky(ax)
    ta, te = np.deg2rad(tgt_az), np.deg2rad(tgt_el)
    tp = (1.02 * np.cos(te) * np.sin(ta), 1.02 * np.cos(te) * np.cos(ta),
          1.02 * np.sin(te))
    ax.plot([0, tp[0]], [0, tp[1]], [0, tp[2]], color=CYAN, lw=1.0, ls=":")
    ax.scatter(*tp, s=120, marker="X", color=RED, depthshade=False)
    ba, be = np.deg2rad(bore), np.deg2rad(tgt_el)
    ax.plot([0, 1.1 * np.cos(be) * np.sin(ba)], [0, 1.1 * np.cos(be) * np.cos(ba)],
            [0, 1.1 * np.sin(be)], color="#f2f5fa", lw=1.4)
    ax.scatter([0], [0], [0], s=60, marker="^", color="#f2f5fa", depthshade=False)
    ax.set_xlabel("east", fontsize=7, labelpad=-6, color=FG)
    ax.set_ylabel("north", fontsize=7, labelpad=-6, color=FG)
    ax.set_zlabel("up", fontsize=7, labelpad=-6, color=FG)
    ax.set_title("two squinted lobes and the aircraft between them",
                 fontsize=9, color=FG)

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    a1.plot(th + bore, dA, color=BLUE, lw=1.4, label="beam A")
    a1.plot(th + bore, dB, color=ORANGE, lw=1.4, label="beam B")
    a1.axvline(tgt_az, color=RED, lw=1.2)
    a1.axvline(bore, color="#f2f5fa", lw=1.0, ls="--")
    ga = gauss_beam(tgt_az - (bore - squint), bw)
    gb = gauss_beam(tgt_az - (bore + squint), bw)
    tot = ga + gb
    acq = tot > 1e-6                      # both channels dead = nothing to track
    ratio = (gb - ga) / tot if acq else np.nan
    a1.plot([tgt_az, tgt_az], [ga, gb], "o", color=RED, ms=6)
    a1.set_xlabel("azimuth  (deg)"); a1.set_ylabel("gain")
    a1.legend(fontsize=7)
    a1.set_title(f"A = {ga:.3e}   B = {gb:.3e}   " +
                 (f"Δ/Σ = {ratio:+.4f}" if acq
                  else "both channels dark — outside the basket"))

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    a2.plot(hist, color=GREEN, lw=1.1)
    a2.axhline(tgt_az, color=RED, lw=1.0, ls="--")
    a2.axvline(kk, color=CYAN, lw=1.0)
    a2.set_xlabel("pulse"); a2.set_ylabel("boresight  (deg)")
    a2.set_title("the loop walks the lobes onto the target")

    err = hist[max(kk - 40, 0):kk + 1] - tgt_az
    readout(fig, 0.845, 0.92, [
        "ANTENNA", "─" * 26,
        f"technique {mode.split()[0]:>16s}",
        f"beamwidth   {bw:>10.2f}°",
        f"squint      {squint:>+9.2f}°",
        f"slope       {slope:>10.3f}/°",
        "", "TARGET", "─" * 26,
        f"az / el     {tgt_az:>+6.2f}/{tgt_el:>+4.0f}°",
        f"boresight   {bore:>+9.3f}°",
        f"error       {bore-tgt_az:>+9.4f}°",
        "", "CHANNELS", "─" * 26,
        f"A           {ga:>10.4f}",
        f"B           {gb:>10.4f}",
        f"Δ/Σ         {ratio:>+10.4f}" if acq else
        "Δ/Σ            no signal",
        "", "ACCURACY", "─" * 26,
        f"SNR         {snr_db:>10.1f}dB",
        f"rms error   {np.std(err):>10.4f}°",
        f"pulse       {kk:>5d}/{len(hist)-1:<5d}",
        "TRACKING" if acq else "TARGET OUTSIDE BEAMS",
    ], color=FG if acq else RED)
    footer(fig, f"{mode}   beamwidth {bw:.1f}°   squint ±{squint:.2f}°   "
                f"loop gain 0.5   {len(hist)} pulses")
    plt.show()


_pL, _sL = timeline(159, desc="pulse")
wL = dict(bw=widgets.FloatSlider(value=6, min=2, max=16, step=0.5,
                                 description="beamwidth:", **SL),
          squint=widgets.FloatSlider(value=1.5, min=0.25, max=6, step=0.25,
                                     description="squint ±:", **SL),
          tgt_az=widgets.FloatSlider(value=8, min=-25, max=25, step=0.5,
                                     description="target az:", **SL),
          tgt_el=widgets.FloatSlider(value=15, min=0, max=45, step=1,
                                     description="target el:", **SL),
          snr_db=widgets.FloatSlider(value=25, min=0, max=45, step=1,
                                     description="SNR (dB):", **SL),
          mode=widgets.Dropdown(options=["monopulse (simultaneous)",
                                         "sequential lobing"],
                                value="monopulse (simultaneous)",
                                description="technique:", **SL),
          k=_sL)
display(widgets.VBox([widgets.HBox([wL["bw"], wL["squint"], wL["tgt_az"]]),
                      widgets.HBox([wL["tgt_el"], wL["snr_db"], wL["mode"]]),
                      widgets.HBox([_pL, _sL])]),
        widgets.interactive_output(draw_lobing3d, wL))

Output()

## Nodding — a pencil beam sweeping a volume with aircraft in it

The beam is a narrow cone and the sky is not. To search a volume the antenna rotates in azimuth while nodding in elevation, and the cone paints one bar at a time.

$$N_{\text{beams}}\approx\frac{\Omega}{\theta_{az}\theta_{el}},\qquad
T_{\text{frame}}=N_{\text{beams}}\,\frac{n_p}{\text{PRF}}$$

The vertical cut is the panel that makes this concrete. Each elevation bar is a wedge reaching out to the detection range, and between the wedges there is nothing — an aircraft flying in a gap is simply not detected, however large its echo. The wedges also fan out with range, so the same 4° bar that is 349 m tall at 5 km is 3492 m tall at 50 km: **coverage is dense close in and full of holes far out**, which is why long-range search radars use fan beams in elevation and accept the loss of height information.

Three aircraft fly through while the beam works. They light up only in the instant the cone is on them, and the gaps between illuminations are the frame time. That interval is also what an intercept receiver measures — the scan pattern is a signature, and a faster search is a louder one.

Halve the beamwidth and you gain 6 dB and lose four times the revisit rate. The aircraft that was detected twice per pass may now be missed entirely.

In [ ]:
def nod_point(k, az_rate, nod_rate, el_lo, el_hi, dt=0.05):
    t = k * dt
    az = ((az_rate * t + 60) % 120) - 60
    span = max(el_hi - el_lo, 1e-6)
    ph = (nod_rate * t / span) % 2
    return az, el_lo + span * (ph if ph < 1 else 2 - ph), t


AC = [dict(name="AC-1", p0=np.array([-45e3, 55e3, 9e3]),
           v=np.array([210.0, -70.0, 0.0])),
      dict(name="AC-2", p0=np.array([30e3, 70e3, 3e3]),
           v=np.array([-160.0, -140.0, 30.0])),
      dict(name="AC-3", p0=np.array([5e3, 30e3, 14e3]),
           v=np.array([60.0, 90.0, -20.0]))]


def ac_state(k, dt=0.05):
    out = []
    for a in AC:
        p = a["p0"] + a["v"] * k * dt
        gr = np.hypot(p[0], p[1])
        out.append(dict(name=a["name"], p=p,
                        az=np.degrees(np.arctan2(p[0], p[1])),
                        el=np.degrees(np.arctan2(p[2], gr)),
                        R=np.linalg.norm(p), gr=gr))
    return out


def draw_nod3d(k, az_rate, nod_rate, bw, el_lo, el_hi, rmax_km):
    az, el, t = nod_point(k, az_rate, nod_rate, el_lo, el_hi)
    st = ac_state(k)
    lit = [a for a in st
           if np.hypot(a["az"] - az, a["el"] - el) < bw / 2 and a["R"] < rmax_km * 1e3]
    ks = np.arange(max(0, k - 260), k + 1)
    trail = np.array([nod_point(i, az_rate, nod_rate, el_lo, el_hi)[:2] for i in ks])

    fig = plt.figure(figsize=(13.2, 5.6))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.2, 1.05, 0.55],
                          height_ratios=[1, 0.8], wspace=0.24, hspace=0.44,
                          left=0.02, right=0.995, top=0.92, bottom=0.09)

    ax = fig.add_subplot(gs[:, 0], projection="3d")
    ax.set_facecolor(PANEL)
    gx, gy = np.meshgrid(np.linspace(-rmax_km, rmax_km, 9),
                         np.linspace(0, rmax_km, 6))
    ax.plot_wireframe(gx, gy, np.zeros_like(gx), color=GRIDC, lw=0.5)
    ca, ce = np.deg2rad(az), np.deg2rad(el)
    L = rmax_km
    circ = np.linspace(0, 2 * np.pi, 26)
    rr = L * np.tan(np.deg2rad(bw / 2))
    u = np.array([np.cos(ce) * np.sin(ca), np.cos(ce) * np.cos(ca), np.sin(ce)])
    e1 = np.array([np.cos(ca), -np.sin(ca), 0.0])
    e2 = np.cross(u, e1)
    for c in circ[::2]:
        p = L * u + rr * (np.cos(c) * e1 + np.sin(c) * e2)
        ax.plot([0, p[0]], [0, p[1]], [0, p[2]], color=CYAN, lw=0.5, alpha=0.5)
    if len(trail):
        ta, te = np.deg2rad(trail[:, 0]), np.deg2rad(trail[:, 1])
        f = np.linspace(0.05, 1.0, len(ta))
        ax.scatter(0.75 * L * np.cos(te) * np.sin(ta),
                   0.75 * L * np.cos(te) * np.cos(ta),
                   0.75 * L * np.sin(te), s=3, c=f, cmap=HOT, depthshade=False)
    for a in st:
        hit = a in lit
        ax.plot([a["p"][0] / 1e3, a["p"][0] / 1e3], [a["p"][1] / 1e3, a["p"][1] / 1e3],
                [0, a["p"][2] / 1e3], color=GRIDC, lw=0.6)
        ax.scatter([a["p"][0] / 1e3], [a["p"][1] / 1e3], [a["p"][2] / 1e3],
                   s=110 if hit else 45, marker="X" if hit else "o",
                   color=RED if hit else ORANGE, depthshade=False)
    ax.scatter([0], [0], [0], s=70, marker="^", color="#f2f5fa", depthshade=False)
    ax.set_xlim(-rmax_km, rmax_km); ax.set_ylim(0, rmax_km)
    ax.set_zlim(0, rmax_km * 0.45)
    ax.set_box_aspect((2.0, 1.0, 0.5)); ax.grid(False)
    ax.tick_params(colors=MUTED, labelsize=6)
    for p in (ax.xaxis, ax.yaxis, ax.zaxis):
        p.set_pane_color((0, 0, 0, 0))
    ax.view_init(elev=24, azim=-64)
    ax.set_xlabel("east (km)", fontsize=7, labelpad=-6, color=FG)
    ax.set_ylabel("north (km)", fontsize=7, labelpad=-6, color=FG)
    ax.set_zlabel("alt (km)", fontsize=7, labelpad=-6, color=FG)
    ax.set_title("the cone paints one bar at a time", fontsize=9, color=FG)

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    nbar = max(int((el_hi - el_lo) / bw), 1)
    for i in range(nbar + 1):
        e0 = el_lo + i * bw
        for sgn in (-0.5, 0.5):
            ee = np.deg2rad(e0 + sgn * bw)
            a1.plot([0, rmax_km * np.cos(ee)], [0, rmax_km * np.sin(ee)],
                    color=GRIDC, lw=0.6)
        arc = np.deg2rad(np.linspace(e0 - bw / 2, e0 + bw / 2, 12))
        a1.fill(np.r_[0, rmax_km * np.cos(arc)], np.r_[0, rmax_km * np.sin(arc)],
                color=CYAN, alpha=0.10)
    arc = np.deg2rad(np.linspace(el - bw / 2, el + bw / 2, 12))
    a1.fill(np.r_[0, rmax_km * np.cos(arc)], np.r_[0, rmax_km * np.sin(arc)],
            color=CYAN, alpha=0.5)
    for a in st:
        a1.plot(a["gr"] / 1e3, a["p"][2] / 1e3, "X" if a in lit else "o",
                ms=10 if a in lit else 6, color=RED if a in lit else ORANGE)
        a1.text(a["gr"] / 1e3, a["p"][2] / 1e3 + 1.0, a["name"], color=FG,
                fontsize=6.5, ha="center")
    a1.set_xlim(0, rmax_km); a1.set_ylim(0, rmax_km * 0.42)
    a1.set_xlabel("ground range  (km)"); a1.set_ylabel("altitude  (km)")
    a1.set_title("vertical cut — the bars, and the gaps between them")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    if len(trail):
        a2.scatter(trail[:, 0], trail[:, 1], s=5,
                   c=np.linspace(0.1, 1, len(trail)), cmap=HOT)
    for a in st:
        a2.plot(a["az"], a["el"], "X" if a in lit else "o",
                ms=9 if a in lit else 5, color=RED if a in lit else ORANGE)
    a2.set_xlim(-62, 62); a2.set_ylim(el_lo - 3, el_hi + 3)
    a2.set_xlabel("azimuth  (deg)"); a2.set_ylabel("elevation  (deg)")
    a2.set_title("where the beam has been")

    frame = 2 * (el_hi - el_lo) / max(nod_rate, 1e-6)
    lines = ["SCAN", "─" * 26,
             f"az rate     {az_rate:>8.1f}°/s",
             f"nod rate    {nod_rate:>8.1f}°/s",
             f"beamwidth   {bw:>10.2f}°",
             f"bars        {nbar:>10d}",
             f"nod period  {frame:>10.2f}s",
             f"time        {t:>10.2f}s", "", "AIRCRAFT", "─" * 26]
    for a in st:
        lines += [f"{a['name']}{'  ILLUM' if a in lit else '':>18s}",
                  f"  R      {a['R']/1e3:>10.1f}km",
                  f"  az/el  {a['az']:>+6.1f}/{a['el']:>+4.1f}°",
                  f"  alt    {a['p'][2]/1e3:>10.1f}km"]
    readout(fig, 0.845, 0.92, lines, size=6.9,
            color=RED if lit else FG)
    footer(fig, f"pencil beam {bw:.1f}°   az ±60°   el {el_lo:.0f}–{el_hi:.0f}°   "
                f"{nbar} bars   dt = 0.05 s")
    plt.show()


_pN, _sN = timeline(499, step=3, desc="time step")
wN = dict(az_rate=widgets.FloatSlider(value=30, min=6, max=120, step=2,
                                      description="az rate °/s:", **SL),
          nod_rate=widgets.FloatSlider(value=22, min=4, max=90, step=2,
                                       description="nod rate °/s:", **SL),
          bw=widgets.FloatSlider(value=5, min=2, max=16, step=0.5,
                                 description="beamwidth:", **SL),
          el_lo=widgets.FloatSlider(value=2, min=0, max=15, step=1,
                                    description="el min:", **SL),
          el_hi=widgets.FloatSlider(value=26, min=10, max=50, step=1,
                                    description="el max:", **SL),
          rmax_km=widgets.FloatSlider(value=90, min=40, max=140, step=10,
                                      description="range scale:", **SL),
          k=_sN)
display(widgets.VBox([widgets.HBox([wN["az_rate"], wN["nod_rate"], wN["bw"]]),
                      widgets.HBox([wN["el_lo"], wN["el_hi"], wN["rmax_km"]]),
                      widgets.HBox([_pN, _sN])]),
        widgets.interactive_output(draw_nod3d, wN))

Output()

## Clutter — the ground the beam is standing on

Clutter is not noise and not a spike. It is a **patch of ground**, and the way to understand it is to draw that patch.

Two families of curves live on the terrain. Constant range from the aircraft draws circles; constant Doppler draws **hyperbolas**, because a ground point at $(x,y,0)$ under a platform at altitude $h$ flying along $x$ returns

$$f_c=\frac{2v}{\lambda}\cdot\frac{x}{\sqrt{x^2+y^2+h^2}}$$

and setting that to a constant gives a cone through the aircraft whose intersection with flat ground is a hyperbola. The Doppler spectrum you measure is nothing more than the beam footprint sliced by those hyperbolas — which is why the ridge is broad and structured rather than a line.

The footprint itself is the other half of the story, and it is violently sensitive to depression angle: a 4° beam covers 0.75 km of ground at 60° depression and **19.3 km** at 10°. Looking down steeply illuminates a small patch and a little clutter; looking out at the horizon floodlights tens of kilometres of terrain into the same range cell.

Directly beneath the aircraft the hyperbolas collapse to $f=0$ — the **altitude line**, a strong return at zero Doppler and minimum range that appears in every airborne radar. A target crossing the beam at 90° has zero Doppler too, and lands right on top of it.

In [ ]:
def draw_clutter_ground(k, v_plat, alt_km, look_az, dep_deg, bw, lam_cm, prf,
                        tgt_v, mti):
    lam = lam_cm / 100
    h = alt_km * 1e3
    az = look_az + 22 * np.sin(2 * np.pi * k / 140)
    fmax = 2 * v_plat / lam

    ext = 70e3
    g = np.linspace(-ext, ext, 260)
    X, Y = np.meshgrid(g, g)
    R = np.sqrt(X ** 2 + Y ** 2 + h ** 2)
    FD = 2 * v_plat * X / (lam * R)

    ca, cd = np.deg2rad(az), np.deg2rad(dep_deg)
    bore = np.array([np.sin(ca) * np.cos(cd), np.cos(ca) * np.cos(cd), -np.sin(cd)])
    V = np.stack([X, Y, np.full_like(X, -h)], -1)
    Vn = V / np.linalg.norm(V, axis=-1, keepdims=True)
    psi = np.degrees(np.arccos(np.clip(Vn @ bore, -1, 1)))
    W = np.exp(-2.773 * (psi / bw) ** 2)

    fbin = np.linspace(-1.15 * fmax, 1.15 * fmax, 420)
    hist, _ = np.histogram(FD.ravel(), bins=np.r_[fbin, fbin[-1] + 1],
                           weights=(W ** 2 / np.maximum(R, 1) ** 3).ravel())
    hist = hist / max(hist.max(), 1e-30)
    side = 3e-4 * np.exp(-0.5 * (fbin / (0.75 * fmax)) ** 2)
    spec = hist + side
    Hm = (np.abs(2 * np.sin(np.pi * fbin / prf)) ** (1 if mti == "2-pulse" else 2)
          if mti != "none" else np.ones_like(fbin))
    Hm = Hm / max(Hm.max(), 1e-9)
    fdt = 2 * tgt_v / lam

    fig = plt.figure(figsize=(13.2, 5.6))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.2, 1.05, 0.55],
                          height_ratios=[1, 0.8], wspace=0.24, hspace=0.44,
                          left=0.02, right=0.995, top=0.92, bottom=0.09)

    ax = fig.add_subplot(gs[:, 0], projection="3d")
    ax.set_facecolor(PANEL)
    sub = slice(None, None, 4)
    ax.plot_surface(X[sub, sub] / 1e3, Y[sub, sub] / 1e3,
                    np.zeros_like(X[sub, sub]),
                    facecolors=FIELD((FD[sub, sub] / fmax * 0.5 + 0.5)),
                    rstride=1, cstride=1, linewidth=0, antialiased=False,
                    shade=False, alpha=0.85)
    lvl = W.max() * np.array([0.5])
    ax.contour(X / 1e3, Y / 1e3, W, levels=lvl, colors=[CYAN], linewidths=1.6,
               offset=0.05)
    ax.plot([0], [0], [alt_km], marker="^", ms=9, color="#f2f5fa")
    ax.plot([0, bore[0] * 60], [0, bore[1] * 60],
            [alt_km, alt_km + bore[2] * 60], color=CYAN, lw=1.2)
    ax.set_xlim(-ext / 1e3, ext / 1e3); ax.set_ylim(-ext / 1e3, ext / 1e3)
    ax.set_zlim(0, alt_km * 2.2)
    ax.set_box_aspect((1.6, 1.6, 0.55)); ax.grid(False)
    ax.tick_params(colors=MUTED, labelsize=6)
    for p in (ax.xaxis, ax.yaxis, ax.zaxis):
        p.set_pane_color((0, 0, 0, 0))
    ax.view_init(elev=42, azim=-62)
    ax.set_xlabel("along track (km)", fontsize=7, labelpad=-6, color=FG)
    ax.set_ylabel("cross track (km)", fontsize=7, labelpad=-6, color=FG)
    ax.set_zlabel("alt (km)", fontsize=7, labelpad=-6, color=FG)
    ax.set_title("ground coloured by Doppler, beam footprint outlined",
                 fontsize=9, color=FG)

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    a1.contour(X / 1e3, Y / 1e3, FD / 1e3, levels=11, colors=[ORANGE],
               linewidths=0.7, alpha=0.8)
    for rk in (10, 20, 30, 40, 50, 60):
        c = np.linspace(0, 2 * np.pi, 200)
        gr = np.sqrt(max((rk * 1e3) ** 2 - h ** 2, 0)) / 1e3
        if gr > 0:
            a1.plot(gr * np.cos(c), gr * np.sin(c), color=GRIDC, lw=0.6)
    a1.contour(X / 1e3, Y / 1e3, W, levels=[W.max() * 0.5], colors=[CYAN],
               linewidths=1.8)
    a1.plot(0, 0, "^", ms=9, color="#f2f5fa")
    a1.annotate("", xy=(14, 0), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color="#f2f5fa", lw=1.3))
    a1.set_xlim(-ext / 1e3, ext / 1e3); a1.set_ylim(-ext / 1e3, ext / 1e3)
    a1.set_aspect("equal"); a1.grid(False)
    a1.set_xlabel("along track  (km)"); a1.set_ylabel("cross track  (km)")
    a1.set_title("iso-Doppler hyperbolas · iso-range circles · footprint")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    out = spec * Hm ** 2
    a2.fill_between(fbin / 1e3, -80, 10 * np.log10(np.maximum(out, 1e-8)),
                    color=ORANGE, alpha=0.30)
    a2.plot(fbin / 1e3, 10 * np.log10(np.maximum(out, 1e-8)), color=ORANGE, lw=1.0)
    tgl = 0.02 * np.exp(-0.5 * ((fbin - fdt) / 90) ** 2) * Hm ** 2
    a2.plot(fbin / 1e3, 10 * np.log10(np.maximum(tgl, 1e-8)), color=GREEN, lw=1.3)
    a2.axvline(0, color=CYAN, lw=0.8, ls=":")
    a2.text(0.3, -70, "altitude line", color=CYAN, fontsize=6.5)
    a2.set_ylim(-80, 3); a2.set_xlabel("Doppler  (kHz)"); a2.set_ylabel("dB")
    a2.set_title("spectrum = the footprint sliced by those hyperbolas")

    rn = h / np.tan(np.deg2rad(dep_deg + bw / 2))
    rf = h / np.tan(np.deg2rad(max(dep_deg - bw / 2, 0.3)))
    vis = 20 * np.log10(max(np.interp(fdt, fbin, Hm), 1e-4))
    readout(fig, 0.845, 0.92, [
        "PLATFORM", "─" * 26,
        f"speed       {v_plat:>9.0f}m/s",
        f"altitude    {alt_km:>10.1f}km",
        f"λ           {lam_cm:>10.1f}cm",
        "", "BEAM", "─" * 26,
        f"azimuth     {az:>+9.1f}°",
        f"depression  {dep_deg:>10.1f}°",
        f"beamwidth   {bw:>10.1f}°",
        f"ground near {rn/1e3:>10.2f}km",
        f"ground far  {rf/1e3:>10.2f}km",
        f"footprint   {(rf-rn)/1e3:>10.2f}km",
        "", "CLUTTER", "─" * 26,
        f"span        ±{fmax/1e3:>8.2f}kHz",
        f"main lobe   {2*v_plat*np.cos(np.deg2rad(az))/lam/1e3:>+9.2f}kHz",
        "", "TARGET", "─" * 26,
        f"v_r         {tgt_v:>+9.0f}m/s",
        f"Doppler     {fdt/1e3:>+9.2f}kHz",
        f"MTI loss    {vis:>+9.1f}dB",
        "IN THE NOTCH" if vis < -12 else "VISIBLE",
    ], color=RED if vis < -12 else GREEN, size=6.9)
    footer(fig, f"v={v_plat:.0f} m/s   alt {alt_km:.1f} km   λ={lam_cm:.1f} cm   "
                f"depression {dep_deg:.0f}°   MTI {mti}")
    plt.show()


_pC, _sC = timeline(139, desc="scan step")
wC = dict(v_plat=widgets.FloatSlider(value=250, min=50, max=600, step=10,
                                     description="platform m/s:", **SL),
          alt_km=widgets.FloatSlider(value=8, min=1, max=15, step=0.5,
                                     description="altitude km:", **SL),
          look_az=widgets.FloatSlider(value=25, min=-80, max=80, step=5,
                                      description="look az:", **SL),
          dep_deg=widgets.FloatSlider(value=20, min=4, max=70, step=2,
                                      description="depression:", **SL),
          bw=widgets.FloatSlider(value=6, min=2, max=16, step=0.5,
                                 description="beamwidth:", **SL),
          lam_cm=widgets.FloatSlider(value=3, min=1, max=30, step=0.5,
                                     description="λ (cm):", **SL),
          prf=widgets.FloatSlider(value=6000, min=1000, max=20000, step=500,
                                  description="PRF (Hz):", **SL),
          tgt_v=widgets.FloatSlider(value=140, min=-400, max=400, step=10,
                                    description="target v_r:", **SL),
          mti=widgets.Dropdown(options=["none", "2-pulse", "3-pulse"],
                               value="2-pulse", description="MTI:", **SL),
          k=_sC)
display(widgets.VBox([widgets.HBox([wC["v_plat"], wC["alt_km"], wC["look_az"]]),
                      widgets.HBox([wC["dep_deg"], wC["bw"], wC["lam_cm"]]),
                      widgets.HBox([wC["prf"], wC["tgt_v"], wC["mti"]]),
                      widgets.HBox([_pC, _sC])]),
        widgets.interactive_output(draw_clutter_ground, wC))

Output()

## Measuring range — watching the pulses actually travel

Range is a stopwatch: send a pulse, wait for the echo, halve the round trip. The picture that matters is the one where you can see the pulses **in flight**, because that is where the ambiguity comes from.

$$R=\frac{c\,t_d}{2},\qquad R_{ua}=\frac{c}{2\,\text{PRF}}$$

At a low PRF only one wavefront is ever airborne, every echo belongs to the pulse that just left, and the reading is honest. Raise the PRF and the transmitter fires again before the far echoes are home — the plan view fills with concentric rings, and the receiver has no way to tell which ring a return came from. It assumes the most recent, so a distant target is reported at a short range.

The numbers are stark. At 2 kHz the unambiguous range is 75 km and T2 at 58 km reads correctly. Raise the PRF to 8 kHz and $R_{ua}$ collapses to 18.7 km: the same aircraft is now reported at **1.8 km**, with three transmits having gone out while its echo was still in the air. Nothing about the target changed.

Velocity is the mirror image. It comes from the phase advance between pulses, so the PRF sets the highest Doppler that can be sampled without folding, and the two ambiguities move in opposite directions with their product pinned at $c\lambda/4$. Drag the PRF and watch the rings multiply while the blind speed climbs.

In [ ]:
C0 = 2.998e8                       # local constant, this cell stands alone

MEAS_TGT = [dict(name="T1", R0=22e3, az=-28.0, v=-110.0, rcs=1.0),
            dict(name="T2", R0=58e3, az=12.0, v=180.0, rcs=0.8),
            dict(name="T3", R0=96e3, az=40.0, v=-60.0, rcs=0.6)]


def draw_measure(k, prf, tau_us, lam_cm, rmax_km, show_ambig):
    lam = lam_cm / 100
    t = k * 4e-6
    Rua = C0 / (2 * prf)
    tg = [dict(g, R=g["R0"] + g["v"] * k * 4e-6 * 300) for g in MEAS_TGT]

    fig = plt.figure(figsize=(13.2, 5.6))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.2, 1.05, 0.55],
                          height_ratios=[1, 0.8], wspace=0.26, hspace=0.46,
                          left=0.05, right=0.995, top=0.92, bottom=0.09)

    a0 = panel(fig.add_subplot(gs[:, 0], projection="polar"), BLUE)
    a0.set_facecolor(PANEL)
    n_air = 0
    kk = 0
    while True:
        r = C0 * (t - kk / prf)
        if r <= 0:
            break
        if r / 1e3 <= rmax_km:
            th = np.linspace(-np.pi / 2, np.pi / 2, 200)
            a0.plot(th, np.full_like(th, r / 1e3), color=CYAN,
                    lw=1.6 if kk == 0 else 0.9, alpha=1.0 if kk == 0 else 0.45)
            n_air += 1
        kk += 1
        if kk > 400:
            break
    for g in tg:
        if g["R"] / 1e3 <= rmax_km:
            a0.plot([np.deg2rad(g["az"])], [g["R"] / 1e3], "o", ms=9, color=ORANGE)
            a0.text(np.deg2rad(g["az"]), g["R"] / 1e3 + rmax_km * 0.05,
                    g["name"], color=FG, fontsize=7, ha="center")
        if show_ambig:
            a0.plot(np.linspace(-np.pi / 2, np.pi / 2, 100),
                    np.full(100, (g["R"] % Rua) / 1e3), color=RED, lw=0.8, ls=":")
    for n in range(1, int(rmax_km * 1e3 / Rua) + 1):
        a0.plot(np.linspace(-np.pi / 2, np.pi / 2, 100),
                np.full(100, n * Rua / 1e3), color=GREEN, lw=0.9, ls="--",
                alpha=0.7)
    a0.set_theta_zero_location("N"); a0.set_theta_direction(-1)
    a0.set_thetamin(-90); a0.set_thetamax(90)
    a0.set_ylim(0, rmax_km); a0.set_rlabel_position(272)
    a0.tick_params(colors=MUTED, labelsize=6.5); a0.grid(alpha=0.2, color=GRIDC)
    a0.set_title(f"{n_air} wavefront(s) in flight — green rings are multiples "
                 f"of $R_{{ua}}$", pad=14)

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    span = 3 / prf
    tt = np.linspace(0, span, 3000)
    tx = np.zeros_like(tt)
    for n in range(4):
        tx += ((tt >= n / prf) & (tt < n / prf + tau_us * 1e-6)).astype(float)
    a1.fill_between(tt * 1e6, 0, tx, color=CYAN, alpha=0.5, step="mid")
    for i, g in enumerate(tg):
        td = 2 * g["R"] / C0
        rxs = np.zeros_like(tt)
        for n in range(4):
            c0 = n / prf + td
            rxs += ((tt >= c0) & (tt < c0 + tau_us * 1e-6)).astype(float)
        a1.fill_between(tt * 1e6, -0.9 - i, -0.9 - i + 0.7 * rxs,
                        color=ORANGE, alpha=0.75, step="mid")
        a1.text(span * 1e6 * 0.995, -0.65 - i, g["name"], color=FG, fontsize=6.5,
                ha="right")
    for n in range(4):
        a1.axvline(n / prf * 1e6, color=GREEN, lw=0.8, ls="--", alpha=0.7)
    a1.set_xlim(0, span * 1e6); a1.set_ylim(-3.6, 1.3); a1.set_yticks([])
    a1.set_xlabel("time  (µs)")
    a1.set_title("transmit (top) and echoes — which pulse do they belong to?")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    tr = np.linspace(0, rmax_km, 500)
    a2.plot(tr, (tr * 1e3 % Rua) / 1e3, color=CYAN, lw=1.5, label="reported")
    a2.plot(tr, tr, color=GRIDC, lw=1.0, ls="--", label="truth")
    for g in tg:
        a2.plot(g["R"] / 1e3, (g["R"] % Rua) / 1e3, "o", ms=8, color=ORANGE)
        a2.text(g["R"] / 1e3, (g["R"] % Rua) / 1e3 + rmax_km * 0.03, g["name"],
                color=FG, fontsize=6.5, ha="center")
    a2.set_xlabel("true range  (km)"); a2.set_ylabel("reported  (km)")
    a2.legend(fontsize=7)
    a2.set_title(f"everything folds into 0–{Rua/1e3:.1f} km")

    lines = ["TIMING", "─" * 26,
             f"PRF         {prf/1e3:>10.2f}kHz",
             f"PRI         {1e6/prf:>10.2f}µs",
             f"τ           {tau_us:>10.2f}µs",
             f"R unamb     {Rua/1e3:>10.2f}km",
             f"in flight   {n_air:>10d}",
             f"v blind     {lam*prf/2:>10.1f}m/s",
             f"product     {Rua/1e3*lam*prf/2:>10.0f}",
             f"cλ/4        {C0*lam/4/1e3:>10.0f}", "", "TARGETS", "─" * 26]
    for g in tg:
        folded = g["R"] > Rua
        lines += [f"{g['name']}{'  FOLDED' if folded else '':>18s}",
                  f"  true   {g['R']/1e3:>10.1f}km",
                  f"  says   {(g['R']%Rua)/1e3:>10.1f}km",
                  f"  v_r    {g['v']:>+10.0f}m/s"]
    readout(fig, 0.845, 0.92, lines, size=6.9)
    footer(fig, f"PRF {prf/1e3:.2f} kHz   R_ua = c/2PRF = {Rua/1e3:.2f} km   "
                f"λ={lam_cm:.1f} cm   v_blind = {lam*prf/2:.0f} m/s")
    plt.show()


_pM, _sM = timeline(160, step=2, desc="time step")
wM = dict(prf=widgets.FloatSlider(value=2000, min=500, max=20000, step=250,
                                  description="PRF (Hz):", **SL),
          tau_us=widgets.FloatSlider(value=8, min=1, max=40, step=1,
                                     description="pulse τ (µs):", **SL),
          lam_cm=widgets.FloatSlider(value=3, min=1, max=30, step=0.5,
                                     description="λ (cm):", **SL),
          rmax_km=widgets.FloatSlider(value=120, min=40, max=200, step=10,
                                      description="display km:", **SL),
          show_ambig=widgets.Checkbox(value=True, description="show folded ranges",
                                      indent=False),
          k=_sM)
display(widgets.VBox([widgets.HBox([wM["prf"], wM["tau_us"], wM["lam_cm"]]),
                      widgets.HBox([wM["rmax_km"], wM["show_ambig"]]),
                      widgets.HBox([_pM, _sM])]),
        widgets.interactive_output(draw_measure, wM))

Output()